# TAK861 — collect the missing experiment evidence

**Auxiliary discovery notebook.** This does not replace any of the four current
model notebooks or the planned fifth comparison notebook. It gathers evidence
needed before the final49/1,028 experiments. It does not train, select features,
score TEST, change splits, call build procedures, or write warehouse tables.

Run in the approved **Databricks Python** environment, using the existing
Snowflake connection setup. Everything needed by this notebook is embedded;
there are no repository-module imports, package installs or model downloads.

Run from top to bottom. Share each printed `EVIDENCE_BEGIN ... EVIDENCE_END`
block with its **E-number and run ID**. Large metadata lists are paginated;
send all pages for the section. Text is preferred; screenshots are acceptable.
The last cell produces a checklist. A failed permission/query is evidence too:
return its sanitized status instead of trying unrelated source replacements.

Outputs contain aggregate results and metadata, not patient rows, secrets,
raw procedure definitions or model weights. Keep the executed notebook in the
work environment; do not commit its outputs. No files are exported automatically.

**Status meanings:** PASS = the named check passed within its stated scope;
READ_OK = readable, not necessarily valid; FAIL = an observed contract violation;
UNAVAILABLE = inaccessible/missing execution dependency; INCONCLUSIVE = insufficient
evidence; SKIPPED = deliberately not executed. None means the whole experiment
is certified leakage-free. Source definitions and catalog timestamps alone cannot
prove what historically ran or whether a holdout was ever used.

**Genie use:** prompts below ask Genie to inspect existing source/audit records.
Ask it to cite exact object, notebook revision, run and relevant source lines.
Do not let it rebuild tables, invent a history or certify an unknown fact.


## E00 — Configuration, safe query/report helpers

**Why:** bind this evidence run to the known V63 objects and prevent accidental
unbounded data collection or unrelated writes. These are source leads, not proof
that the currently visible objects reproduce the historical run.

**Action:** run your existing approved connection setup in this same notebook
session, then run this cell. It reuses `sf_options_dl_poc` or `sf_options` without
printing either. If absent, subsequent warehouse sections report UNAVAILABLE;
you can still return runtime and human-input outputs. Do not paste credentials
in notebook source. Edit only verified object overrides below if a catalog lookup
shows a different binding. Leave `FULL_TENSOR_AUDIT=False` on the first pass.

**Output:** run identity and selected limits. Row limits bound returned data,
not warehouse compute time; even aggregate checks may scan tables. Cancel a slow
query in the work UI and return the affected E-number rather than waiting blindly.

**Genie prompt:** “Reuse the existing approved Snowflake connector setup for this
Databricks notebook without printing or hard-coding secrets. Identify which of
sf_options_dl_poc/sf_options is configured and whether Spark is available.”


In [ ]:
import base64
import hashlib
import io
import json
import math
import os
import platform
import re
import time
import uuid
from datetime import date, datetime, timezone
from decimal import Decimal
from importlib import metadata as package_metadata
import numpy as np
import pandas as pd

EVIDENCE_VERSION = "2026-10-05.v1"
EVIDENCE_RUN_ID = "TAK861_EVIDENCE_" + uuid.uuid4().hex[:12]
FULL_TENSOR_AUDIT = False
READ_SAFE_ARTIFACT_REPORTS = True
MAX_ARTIFACT_BYTES = 2_000_000
MAX_ARTIFACT_TABLES = 12
MAX_REPORT_PAGE_ROWS = 30
PRIVATE_DB = "DSVC_TAKEDA_TA_PRIVATE"
PRIVATE_SCHEMA = "DS_ML"
V63_PREFIX = "TAK861_TX_READY_V63_"
_v63 = PRIVATE_DB + "." + PRIVATE_SCHEMA + "." + V63_PREFIX
tables = {
    "snapshots": _v63 + "DL_POC_SNAPSHOTS",
    "frozen_split": _v63 + "DL_POC_PATIENT_SPLIT",
    "feature_map": _v63 + "DL_POC_FEATURE_MAP",
    "tensor_monthly": _v63 + "DL_POC_TENSOR_MONTHLY",
    "model_data": _v63 + "MODEL_DATA",
    "model_type": _v63 + "MODEL_TYPE",
    "final_model": _v63 + "FINAL_MODEL",
    "features_summary": _v63 + "FEATURES_SUMMARY",
    "universe_features": _v63 + "UNIVERSE_W_FEATURES",
    "mapping": _v63 + "MAPPING",
    "model_comparison": _v63 + "MODEL_COMPARISON",
    "universe_scored": _v63 + "UNIVERSE_W_FEATURES_SCORED",
    "backtest_scored": _v63 + "BACK_TESTING_SCORED",
    "patients_scored": _v63 + "PATIENTS_SCORED",
    "enrollment": PRIVATE_DB + ".DS_ML.ENROLLMENT_20260825",
    "hcp_lookup": PRIVATE_DB + ".DS_ML.HCP_LOOKUP",
    "medical_dated": "DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_20260825",
    "pharmacy_dated": "DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_20260825",
    "dx_mapping": None,  # DX_CSSR_LATEST: resolve a unique visible FQN in E02, never guess.
}
# Optional verified replacements only. Never silently substitute *_LATEST for the dated inputs.
TABLE_OVERRIDES = {}
tables.update(TABLE_OVERRIDES)
CATALOG_DATABASES = [PRIVATE_DB, "DSVC_TAKEDA_FULLMAP_PLAID_PROD"]
reports = []
query_audit = []
private_errors = {}  # Work-session debugging only. Never print/export this dictionary.
schemas = {}
discovered_tables = pd.DataFrame(columns=["TABLE_CATALOG", "TABLE_SCHEMA", "TABLE_NAME", "ROW_COUNT", "CREATED", "LAST_ALTERED"])

class EvidenceUnavailable(RuntimeError):
    pass

class LimitExceeded(RuntimeError):
    pass

def qi(identifier):
    parts = str(identifier).split(".")
    if not 1 <= len(parts) <= 3 or any(not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", x) for x in parts):
        raise ValueError("Use a verified ordinary unquoted Snowflake identifier.")
    return ".".join('"' + x.upper() + '"' for x in parts)

def lit(value):
    return "'" + str(value).replace("'", "''") + "'"

def _read_only_sql(sql):
    # Internal SELECT templates only; never execute source/procedure text or Genie-generated SQL here.
    without_literals = re.sub(r"'(?:''|[^'])*'", "''", str(sql))
    without_comments = re.sub(r"/\*.*?\*/|--[^\n]*", " ", without_literals, flags=re.S)
    if ";" in without_comments or not re.match(r"^\s*(SELECT|WITH)\b", without_comments, re.I):
        raise ValueError("Evidence queries must be a single SELECT/WITH statement.")
    if re.search(r"\b(INSERT|UPDATE|DELETE|MERGE|CREATE|ALTER|DROP|TRUNCATE|CALL|COPY|PUT|GET|GRANT|REVOKE|EXECUTE|REMOVE|UNDROP)\b", without_comments, re.I):
        raise ValueError("A non-read operation was rejected.")
    return str(sql).strip()

def _connection_options():
    name = next((n for n in ("sf_options_dl_poc", "sf_options")
                 if isinstance(globals().get(n), dict) and globals()[n]), None)
    if name is None or "spark" not in globals():
        raise EvidenceUnavailable("Run the existing approved connection setup in this session.")
    opts = dict(globals()[name])
    opts.update(sfDatabase=PRIVATE_DB, sfSchema=PRIVATE_SCHEMA)
    return opts

def run_query(sql, max_rows=1000):
    sql = _read_only_sql(sql)
    if not isinstance(max_rows, int) or not 1 <= max_rows <= 25000:
        raise ValueError("Invalid bounded result size.")
    started = time.monotonic()
    item = {"sql_sha256": hashlib.sha256(sql.encode()).hexdigest(), "started_at_utc": datetime.now(timezone.utc).isoformat()}
    try:
        options = _connection_options()
        frame = (spark.read.format("snowflake").options(**options)
                 .option("query", sql).load().limit(max_rows + 1).toPandas())
        if len(frame) > max_rows:
            raise LimitExceeded("Result exceeds configured bound; narrow the requested inventory.")
        frame.columns = [str(c).upper() for c in frame.columns]
        if frame.columns.duplicated().any():
            raise ValueError("Ambiguous case-normalized result columns.")
        item.update(status="READ_OK", rows=len(frame))
        return frame
    except Exception as error:
        item.update(status="UNAVAILABLE", error_class=type(error).__name__)
        raise
    finally:
        item["elapsed_seconds"] = round(time.monotonic() - started, 3)
        query_audit.append(item)

def columns(fqn):
    if not fqn:
        return set()
    if fqn not in schemas:
        parts = str(fqn).split(".")
        if len(parts) != 3:
            raise ValueError("Column discovery requires a fully qualified table.")
        frame = run_query("SELECT COLUMN_NAME, DATA_TYPE, ORDINAL_POSITION FROM " + qi(parts[0])
                          + ".INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA=" + lit(parts[1].upper())
                          + " AND TABLE_NAME=" + lit(parts[2].upper()) + " ORDER BY ORDINAL_POSITION", 3000)
        schemas[fqn] = frame.to_dict("records")
    return {str(row["COLUMN_NAME"]).upper() for row in schemas[fqn]}

def _json_safe(value):
    if isinstance(value, pd.DataFrame):
        return _json_safe(value.to_dict("records"))
    if isinstance(value, dict):
        denied = {"PATIENT_ID", "MEMBER_ID", "PAYLOAD_BASE64", "PASSWORD", "TOKEN", "SFPASSWORD", "SFURL", "SFUSER", "PRIVATE_KEY", "PROCEDURE_DEFINITION", "QUERY_TEXT"}
        if denied.intersection(str(k).upper() for k in value):
            raise ValueError("Raw or sensitive fields are not permitted in shareable reports.")
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_json_safe(v) for v in value]
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float, Decimal)):
        x = float(value)
        return x if math.isfinite(x) else None
    if isinstance(value, (date, datetime, pd.Timestamp)):
        return value.isoformat()
    if isinstance(value, (str, int, bool)):
        return value
    raise TypeError("Unsupported report value type.")

def emit(section, status, summary, tables=None, details=None):
    allowed = {"PASS", "FAIL", "READ_OK", "INCONCLUSIVE", "UNAVAILABLE", "SKIPPED", "USER_REPORTED"}
    if status not in allowed:
        raise ValueError("Undocumented evidence status.")
    payload = _json_safe({"section": section, "status": status, "summary": summary,
                          "details": details or {}, "tables": tables if tables is not None else []})
    payload.update(run_id=EVIDENCE_RUN_ID, evidence_version=EVIDENCE_VERSION, observed_at_utc=datetime.now(timezone.utc).isoformat())
    # A rerun replaces its section record; restart at E00 after configuration changes.
    reports[:] = [r for r in reports if r["section"] != section]
    reports.append(payload)
    # Paginate records; all pages are printed, no silent head()/rounding/truncation.
    table_sets = payload.pop("tables")
    if isinstance(table_sets, list):
        table_sets = {"records": table_sets}
    elif not isinstance(table_sets, dict):
        raise TypeError("tables must be records, a DataFrame or a named mapping of records.")
    pages = []
    for name, rows in table_sets.items():
        if not isinstance(rows, list):
            rows = [rows]
        for offset in range(0, len(rows), MAX_REPORT_PAGE_ROWS):
            pages.append({"name": name, "row_start": offset, "total_rows": len(rows), "records": rows[offset:offset+MAX_REPORT_PAGE_ROWS]})
    if not pages:
        pages = [{"name": "records", "total_rows": 0, "records": []}]
    payload["tables"] = table_sets
    for number, page in enumerate(pages, 1):
        printable = {k: v for k, v in payload.items() if k != "tables"}
        printable.update(page=number, pages=len(pages), table=page)
        print("EVIDENCE_BEGIN " + str(section) + " " + EVIDENCE_RUN_ID)
        print(json.dumps(printable, indent=2, ensure_ascii=False, allow_nan=False))
        print("EVIDENCE_END " + str(section))
    return payload

def attempt(section, fn):
    try:
        return fn()
    except Exception as error:
        # Do not print connector exception messages: they can contain queries, URLs or identifiers.
        private_errors[section] = error
        return emit(section, "UNAVAILABLE", "This evidence branch did not complete; other branches can still run.",
                    details={"error_class": type(error).__name__, "next_action": "Genie can inspect private_errors for this section in the work session. Do not print/export raw errors or substitute a source silently."})

emit("E00", "PASS", "Evidence helpers configured; no warehouse operation has run.", details={
    "full_tensor_audit": FULL_TENSOR_AUDIT, "safe_artifact_reports": READ_SAFE_ARTIFACT_REPORTS,
    "max_artifact_bytes": MAX_ARTIFACT_BYTES, "max_artifact_tables": MAX_ARTIFACT_TABLES,
    "note": "Metadata/read checks are not proof of historical source execution or leakage safety."})


## E01 — Runtime and connection readiness

**Why:** choose a feasible later tuning budget and distinguish code problems from
missing packages/warehouse access. This cell reports hardware and package versions,
then performs a one-row constant query. It prints no connection settings.

**Interpretation:** CUDA=false is acceptable. READ_OK confirms only this connection
query; individual table access is checked later. A package marked unavailable means
the work environment needs its approved installation process, not a new credential.

**Human input later:** maximum practical session duration cannot be inferred from RAM.


In [ ]:
def runtime_evidence():
    info = {"python": platform.python_version(), "cpu_count": os.cpu_count(),
            "spark_available": "spark" in globals(), "packages": {}}
    for package in ("numpy", "pandas", "scipy", "scikit-learn", "torch", "lightgbm", "pyspark"):
        try:
            info["packages"][package] = package_metadata.version(package)
        except package_metadata.PackageNotFoundError:
            info["packages"][package] = "not installed"
    try:
        import psutil
        info["driver_ram_gib"] = round(psutil.virtual_memory().total / 1024**3, 2)
    except Exception as error:
        info["ram_probe_error_class"] = type(error).__name__
    try:
        import torch
        info["cuda_available"] = torch.cuda.is_available()
        info["gpus"] = [{"name": torch.cuda.get_device_name(i), "ram_gib": round(torch.cuda.get_device_properties(i).total_memory/1024**3, 2)}
                        for i in range(torch.cuda.device_count())]
    except Exception as error:
        info["torch_probe_error_class"] = type(error).__name__
    emit("E01.runtime", "READ_OK", "Local driver/runtime metadata only; training has not run.", details=info)
    probe = run_query("SELECT 1 AS CONNECTION_OK, CURRENT_TIMESTAMP() AS WAREHOUSE_OBSERVED_AT", 1)
    emit("E01.connection", "READ_OK", "Snowflake read query completed.", tables=probe)

attempt("E01", runtime_evidence)


## E02 — Catalog inventory and exact source bindings

**Why:** recover existing tables/artifacts and schemas instead of asking for names
already present in the work catalog. Queries are limited to known project prefixes
and source-name leads in two known databases. This is not a scan of every account.

**Output:** discovered table metadata and a schema summary for configured inputs.
ROW_COUNT is catalog metadata, not the cohort validation in E04. CREATED is table
creation time, not a training run date. An absent catalog row may mean hidden role
permissions. DX_CSSR_LATEST is bound only if exactly one candidate is visible;
multiple candidates remain unresolved.

**Genie prompt:** “For any missing/ambiguous binding in E02, locate the exact object
used by the September2026 TAK861 run. Cite its database/schema/table and source
revision. Do not substitute *_LATEST for 20260825 or assume a table creation date
is the experiment execution date.”


In [ ]:
def inventory_evidence():
    global discovered_tables
    frames = []
    wanted = sorted({v.split(".")[-1].upper() for v in tables.values() if v} | {"DX_CSSR_LATEST"})
    for database in CATALOG_DATABASES:
        def collect_one(database=database):
            frame = run_query("SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, ROW_COUNT, CREATED, LAST_ALTERED "
                + "FROM " + qi(database) + ".INFORMATION_SCHEMA.TABLES WHERE "
                + "(STARTSWITH(TABLE_NAME, " + lit(V63_PREFIX) + ") OR TABLE_NAME IN ("
                + ",".join(lit(x) for x in wanted) + ")) ORDER BY TABLE_SCHEMA, TABLE_NAME", 2500)
            frames.append(frame)
            emit("E02.catalog." + database, "READ_OK", "Visible table metadata; creation times are not run dates.", tables=frame)
        attempt("E02.catalog." + database, collect_one)
    discovered_tables = pd.concat(frames, ignore_index=True) if frames else discovered_tables.iloc[0:0].copy()
    dx = discovered_tables.loc[discovered_tables.TABLE_NAME.eq("DX_CSSR_LATEST")]
    candidates = sorted(set(".".join(str(x) for x in row) for row in dx[["TABLE_CATALOG", "TABLE_SCHEMA", "TABLE_NAME"]].itertuples(index=False, name=None)))
    if not tables.get("dx_mapping") and len(candidates) == 1:
        tables["dx_mapping"] = candidates[0]
    emit("E02.mapping", "READ_OK" if tables.get("dx_mapping") else "INCONCLUSIVE",
         "Visible DX mapping binding; actual historical use remains unknown.", details={"binding": tables.get("dx_mapping"), "candidates": candidates})
    for role, fqn in tables.items():
        def schema_one(role=role, fqn=fqn):
            if not fqn:
                return emit("E02.schema."+role, "UNAVAILABLE", "No unique verified binding is configured.")
            names = columns(fqn)
            structural = [r for r in schemas[fqn] if not re.fullmatch(r"F[0-9]{4}", str(r["COLUMN_NAME"]))]
            emit("E02.schema."+role, "READ_OK" if names else "UNAVAILABLE", "Visible column contract, not SELECT permission or lineage proof.",
                 tables=structural, details={"source": fqn, "column_count": len(names),
                 "tensor_alias_columns": sum(bool(re.fullmatch(r"F[0-9]{4}", x)) for x in names),
                 "schema_sha256": hashlib.sha256(json.dumps(_json_safe(schemas[fqn]),sort_keys=True).encode()).hexdigest()})
        attempt("E02.schema."+role, schema_one)

attempt("E02", inventory_evidence)


## E03 — Read access to the configured inputs

**Why:** catalog visibility does not imply readable rows. A constant probe tests
each exact binding without returning records. It also distinguishes an empty
object from a populated one. These probes do not validate cohort counts.

**Interpretation:** READ_OK + nonempty=false means accessible but empty. UNAVAILABLE
does not establish that a table does not exist. Return the source/status and use
Genie to identify the existing permitted object; do not request role changes here.


In [ ]:
def probe_inputs():
    for role, fqn in tables.items():
        def one(role=role, fqn=fqn):
            if not fqn:
                return emit("E03."+role, "UNAVAILABLE", "Unbound source; not probed.")
            frame = run_query("SELECT 1 AS ACCESS_OK FROM " + qi(fqn) + " LIMIT 1", 1)
            emit("E03."+role, "READ_OK", "Exact source is readable; no row values were returned.",
                 details={"source": fqn, "nonempty_probe": len(frame) == 1})
        attempt("E03."+role, one)

attempt("E03", probe_inputs)


## E04 — Frozen population and split integrity

**Why:** All final comparisons must use the same patient-snapshot identities and labels. Counts alone cannot establish this. This cell checks key uniqueness, exact two-way key agreement, RESP agreement, patient overlap, and the frozen split counts. No patient identifiers or row-level labels are printed.

**Output:** Separate population, split, and alignment reports. `PASS` means these specific live-table checks passed. It does **not** prove that upstream feature fitting used TRAIN only or that TEST has never been inspected. `FAIL` means a discrepancy must be resolved before training; `UNAVAILABLE` means the query could not be completed.

**Genie help if needed:** “Read only: locate the frozen V63 DL_POC_SNAPSHOTS and DL_POC_PATIENT_SPLIT used by the existing four notebooks. Explain any schema/count/key discrepancies. Do not regenerate the population or split, modify RESP, or expose patient rows. Return aggregate diagnostics and the exact source table names.”


In [ ]:
import hashlib
import json
import re

def evidence_require_columns(fqn, required):
    missing = sorted(set(required) - set(columns(fqn)))
    if missing:
        raise ValueError('Required schema columns are unavailable: ' + ', '.join(missing))
    return qi(fqn)

def evidence_aggregate(fqn):
    t = evidence_require_columns(fqn, {'PATIENT_ID', 'END_DT', 'RESP'})
    return run_query(f"""SELECT COUNT(*) AS SNAPSHOTS, COUNT(DISTINCT PATIENT_ID) AS PATIENTS,
        COALESCE(SUM(IFF(RESP = 1, 1, 0)), 0) AS POSITIVES,
        COALESCE(SUM(IFF(RESP = 0, 1, 0)), 0) AS NEGATIVES,
        COALESCE(SUM(IFF(RESP IS NULL OR RESP NOT IN (0, 1), 1, 0)), 0) AS INVALID_LABELS,
        COALESCE(SUM(IFF(PATIENT_ID IS NULL OR END_DT IS NULL, 1, 0)), 0) AS NULL_KEYS,
        COALESCE(SUM(IFF(TRIM(TO_VARCHAR(PATIENT_ID)) = '', 1, 0)), 0) AS BLANK_PATIENT_KEYS,
        MIN(END_DT) AS FIRST_SNAPSHOT_DATE, MAX(END_DT) AS LAST_SNAPSHOT_DATE,
        COUNT(*) - (SELECT COUNT(*) FROM (SELECT PATIENT_ID, END_DT FROM {t} GROUP BY 1,2)) AS DUPLICATE_KEY_EXCESS
        FROM {t}""", max_rows=1).iloc[0].to_dict()

def evidence_population():
    r = evidence_aggregate(tables['snapshots'])
    expected = {'SNAPSHOTS': 23151, 'PATIENTS': 12447, 'POSITIVES': 1345, 'NEGATIVES': 21806,
                'INVALID_LABELS': 0, 'NULL_KEYS': 0, 'BLANK_PATIENT_KEYS': 0, 'DUPLICATE_KEY_EXCESS': 0}
    checks = {key: int(r[key]) == value for key, value in expected.items()}
    emit('E04_POPULATION', 'PASS' if all(checks.values()) else 'FAIL',
         'Live frozen population versus the existing notebook contract.',
         tables=[tables['snapshots']], details={'actual': r, 'expected': expected, 'checks': checks})

def evidence_split():
    t = evidence_require_columns(tables['frozen_split'], {'PATIENT_ID', 'END_DT', 'RESP', 'SPLIT'})
    aggregate = evidence_aggregate(tables['frozen_split'])
    groups = run_query(f"""SELECT CASE WHEN SPLIT IN ('train','validation','test') THEN SPLIT
        ELSE '__INVALID__' END AS SPLIT, COUNT(*) AS SNAPSHOTS, COUNT(DISTINCT PATIENT_ID) AS PATIENTS,
        COALESCE(SUM(IFF(RESP = 1, 1, 0)),0) AS POSITIVES FROM {t} GROUP BY 1 ORDER BY 1""", max_rows=4)
    overlap = run_query(f"""SELECT COUNT(*) AS PATIENTS_IN_MULTIPLE_SPLITS FROM (
        SELECT PATIENT_ID FROM {t} GROUP BY PATIENT_ID HAVING COUNT(DISTINCT SPLIT) > 1)""", max_rows=1)
    expected = {'train': [16256,8712,941], 'validation': [3481,1867,202], 'test': [3414,1868,202]}
    actual = {r['SPLIT']: [int(r['SNAPSHOTS']),int(r['PATIENTS']),int(r['POSITIVES'])]
              for r in groups.to_dict('records')}
    checks = {'split_counts_exact': actual == expected,
              'unique_snapshot_keys': int(aggregate['DUPLICATE_KEY_EXCESS']) == 0,
              'no_null_keys': int(aggregate['NULL_KEYS']) == 0,
              'no_blank_patient_keys': int(aggregate['BLANK_PATIENT_KEYS']) == 0,
              'binary_nonnull_labels': int(aggregate['INVALID_LABELS']) == 0,
              'patient_disjoint': int(overlap.iloc[0]['PATIENTS_IN_MULTIPLE_SPLITS']) == 0}
    emit('E04_SPLIT', 'PASS' if all(checks.values()) else 'FAIL',
         'Frozen split counts and patient separation; no split was created or changed.',
         tables=[tables['frozen_split']], details={'checks': checks, 'actual': actual, 'expected': expected,
         'aggregate': aggregate, 'patient_overlap': overlap.to_dict('records'),
         'test_untouched_status': 'NOT_ESTABLISHED_BY_THIS_AUDIT'})

def evidence_key_alignment(other_key, section):
    a = evidence_require_columns(tables['snapshots'], {'PATIENT_ID','END_DT','RESP'})
    b = evidence_require_columns(tables[other_key], {'PATIENT_ID','END_DT','RESP'})
    # Aggregate each key first: duplicates and mixed labels cannot be hidden by joining on RESP.
    r = run_query(f"""WITH a AS (SELECT PATIENT_ID, END_DT, COUNT(*) AS N, MIN(RESP) AS RESP,
          COUNT(DISTINCT RESP) AS LABELS, SUM(IFF(RESP IS NULL,1,0)) AS NULL_LABELS FROM {a} GROUP BY 1,2),
        b AS (SELECT PATIENT_ID, END_DT, COUNT(*) AS N, MIN(RESP) AS RESP,
          COUNT(DISTINCT RESP) AS LABELS, SUM(IFF(RESP IS NULL,1,0)) AS NULL_LABELS FROM {b} GROUP BY 1,2)
        SELECT COALESCE(SUM(IFF(a.N IS NOT NULL AND b.N IS NULL,1,0)),0) AS MISSING_OTHER_KEYS,
          COALESCE(SUM(IFF(a.N IS NULL AND b.N IS NOT NULL,1,0)),0) AS EXTRA_OTHER_KEYS,
          COALESCE(SUM(IFF(a.N IS NOT NULL AND b.N IS NOT NULL AND a.RESP IS DISTINCT FROM b.RESP,1,0)),0) AS LABEL_MISMATCH_KEYS,
          COALESCE(SUM(IFF(a.N > 1,1,0)),0) AS DUPLICATE_SNAPSHOT_KEYS,
          COALESCE(SUM(IFF(b.N > 1,1,0)),0) AS DUPLICATE_OTHER_KEYS,
          COALESCE(SUM(IFF(a.LABELS <> 1 OR a.NULL_LABELS > 0,1,0)),0) AS BAD_SNAPSHOT_LABEL_KEYS,
          COALESCE(SUM(IFF(b.LABELS <> 1 OR b.NULL_LABELS > 0,1,0)),0) AS BAD_OTHER_LABEL_KEYS
        FROM a FULL OUTER JOIN b ON a.PATIENT_ID=b.PATIENT_ID AND a.END_DT=b.END_DT""", max_rows=1).iloc[0].to_dict()
    emit(section, 'PASS' if all(int(v) == 0 for v in r.values()) else 'FAIL',
         'Exact two-way (PATIENT_ID, END_DT) and RESP alignment with frozen snapshots.',
         tables=[tables['snapshots'], tables[other_key]], details=r)

attempt('E04_POPULATION', evidence_population)
attempt('E04_SPLIT', evidence_split)
attempt('E04_SPLIT_ALIGNMENT', lambda: evidence_key_alignment('frozen_split', 'E04_SPLIT_ALIGNMENT'))
attempt('E04_MODEL_DATA_ALIGNMENT', lambda: evidence_key_alignment('model_data', 'E04_MODEL_DATA_ALIGNMENT'))


## E04B — Snapshot calendar, repetition, and follow-up timing evidence

**Why:** Temporal sampling, repeated snapshots per patient, and split prevalence can explain model instability or expose a mismatch with the intended design. These are population diagnostics only; no predictions, feature associations or tuning decisions are made here. Positive snapshots before a patient's last snapshot are counted for investigation, not automatically treated as errors.

**Output:** END_DT day-of-month counts; month-by-split snapshot, patient and positive counts; a 1/2/3/4/5+ snapshots-per-patient histogram; summary spacing between each patient's distinct snapshot dates; and the count of positive snapshots before that patient's latest snapshot. Calendar follow-up compares the latest END_DT plus 90 days with the **documented** extract date 2026-08-25. Even if the calendar comparison fits, actual label maturity is unresolved until source completeness, event availability and label-generation rules are verified. No raw patient records are printed.

**Genie help if needed:** “Read only: explain the V63 snapshot schedule, any 40-day lag or other offsets, whether patients can contribute positive snapshots before their latest snapshot, and how 90-day RESP follow-up completeness was established for the frozen 20260825 extract. Return generating code/version or documented rules, not inferred patient histories. Do not change dates or RESP.”


In [ ]:
def evidence_snapshot_calendar():
    s = evidence_require_columns(tables['snapshots'], {'PATIENT_ID','END_DT','RESP'})
    day_counts = run_query(f"""SELECT DAY(END_DT) AS DAY_OF_MONTH, COUNT(*) AS SNAPSHOTS,
        COUNT(DISTINCT PATIENT_ID) AS PATIENTS, COALESCE(SUM(IFF(RESP=1,1,0)),0) AS POSITIVES
        FROM {s} GROUP BY 1 ORDER BY 1""", max_rows=32)
    calendar = run_query(f"""SELECT MIN(END_DT) AS FIRST_END_DT, MAX(END_DT) AS LAST_END_DT,
        DATEADD(day,90,MAX(END_DT)) AS LAST_END_DT_PLUS_90,
        TO_DATE('2026-08-25') AS DOCUMENTED_EXTRACT_DATE,
        DATEADD(day,90,MAX(END_DT)) <= TO_DATE('2026-08-25') AS CALENDAR_WINDOW_ENDS_BY_DOCUMENTED_EXTRACT,
        COALESCE(SUM(IFF(DATEADD(day,90,END_DT)>TO_DATE('2026-08-25'),1,0)),0) AS SNAPSHOTS_CALENDAR_WINDOW_AFTER_DOCUMENTED_EXTRACT
        FROM {s}""", max_rows=1)
    emit('E04B_CALENDAR','READ_OK','Observed snapshot schedule and calendar follow-up evidence only.',
         tables={'sources':[tables['snapshots']],'day_of_month':day_counts.to_dict('records')},
         details={'calendar':calendar.to_dict('records'),
                  'documented_extract_date_is_not_verified_data_completeness':True,
                  'label_maturity_status':'UNRESOLVED: dates alone cannot establish complete available 90-day outcomes'})

def evidence_monthly_split_profile():
    s = evidence_require_columns(tables['snapshots'], {'PATIENT_ID','END_DT','RESP'})
    f = evidence_require_columns(tables['frozen_split'], {'PATIENT_ID','END_DT','SPLIT'})
    # Collapse split keys first so bad duplicates cannot multiply population counts.
    profile = run_query(f"""WITH split_keys AS (
        SELECT PATIENT_ID, END_DT, COUNT(*) AS N, MIN(SPLIT) AS SPLIT
        FROM {f} GROUP BY 1,2), joined AS (
        SELECT s.PATIENT_ID, s.END_DT, s.RESP,
          CASE WHEN f.N IS NULL THEN '__MISSING_SPLIT__'
               WHEN f.N<>1 THEN '__DUPLICATE_SPLIT_KEY__'
               WHEN f.SPLIT IN ('train','validation','test') THEN f.SPLIT
               ELSE '__INVALID_SPLIT__' END AS SPLIT
        FROM {s} s LEFT JOIN split_keys f ON s.PATIENT_ID=f.PATIENT_ID AND s.END_DT=f.END_DT)
        SELECT DATE_TRUNC('month',END_DT) AS SNAPSHOT_MONTH, SPLIT,
          COUNT(*) AS SNAPSHOTS, COUNT(DISTINCT PATIENT_ID) AS PATIENTS,
          COALESCE(SUM(IFF(RESP=1,1,0)),0) AS POSITIVES,
          COALESCE(SUM(IFF(RESP=0,1,0)),0) AS NEGATIVES,
          COALESCE(SUM(IFF(RESP IS NULL OR RESP NOT IN (0,1),1,0)),0) AS INVALID_LABELS,
          SUM(IFF(RESP=1,1,0))*1.0/NULLIF(COUNT(*),0) AS OBSERVED_POSITIVE_RATE
        FROM joined GROUP BY 1,2 ORDER BY 1,2""", max_rows=600)
    emit('E04B_MONTH_SPLIT','READ_OK','Observed cohort prevalence by month and frozen split; no model evaluation.',
         tables={'sources':[tables['snapshots'],tables['frozen_split']], 'month_split_profile':profile.to_dict('records')},
         details={'maximum_returned_groups':600,
                  'interpretation':'Differences are diagnostic evidence, not proof of leakage or a reason to revise the frozen split.'})

def evidence_patient_snapshot_repetition():
    s = evidence_require_columns(tables['snapshots'], {'PATIENT_ID','END_DT','RESP'})
    histogram = run_query(f"""WITH per_patient AS (
        SELECT PATIENT_ID, COUNT(*) AS N FROM {s} GROUP BY 1)
        SELECT CASE WHEN N>=5 THEN '5+' ELSE TO_VARCHAR(N) END AS SNAPSHOTS_PER_PATIENT_BUCKET,
          COUNT(*) AS PATIENTS, SUM(N) AS SNAPSHOTS
        FROM per_patient GROUP BY 1 ORDER BY 1""", max_rows=5)
    spacing = run_query(f"""WITH unique_dates AS (
        SELECT DISTINCT PATIENT_ID, END_DT FROM {s} WHERE PATIENT_ID IS NOT NULL AND END_DT IS NOT NULL),
        ordered AS (SELECT PATIENT_ID, END_DT,
          LAG(END_DT) OVER (PARTITION BY PATIENT_ID ORDER BY END_DT) AS PREVIOUS_END_DT FROM unique_dates),
        gaps AS (SELECT DATEDIFF(day,PREVIOUS_END_DT,END_DT) AS GAP_DAYS FROM ordered WHERE PREVIOUS_END_DT IS NOT NULL)
        SELECT COUNT(*) AS ADJACENT_DISTINCT_SNAPSHOT_PAIRS,
          MIN(GAP_DAYS) AS MIN_GAP_DAYS, MAX(GAP_DAYS) AS MAX_GAP_DAYS,
          AVG(GAP_DAYS) AS MEAN_GAP_DAYS, MEDIAN(GAP_DAYS) AS MEDIAN_GAP_DAYS,
          COALESCE(SUM(IFF(GAP_DAYS<30,1,0)),0) AS PAIRS_UNDER_30_DAYS,
          COALESCE(SUM(IFF(GAP_DAYS<=90,1,0)),0) AS PAIRS_WITHIN_90_DAYS
        FROM gaps""", max_rows=1)
    positive_timing = run_query(f"""WITH ordered AS (
        SELECT PATIENT_ID, END_DT, RESP, MAX(END_DT) OVER (PARTITION BY PATIENT_ID) AS LATEST_END_DT FROM {s})
        SELECT COALESCE(SUM(IFF(RESP=1,1,0)),0) AS POSITIVE_SNAPSHOTS,
          COALESCE(SUM(IFF(RESP=1 AND END_DT<LATEST_END_DT,1,0)),0) AS POSITIVES_BEFORE_PATIENT_LATEST_SNAPSHOT,
          COUNT(DISTINCT IFF(RESP=1 AND END_DT<LATEST_END_DT,PATIENT_ID,NULL)) AS PATIENTS_WITH_POSITIVES_BEFORE_LATEST
        FROM ordered""", max_rows=1)
    emit('E04B_REPETITION','READ_OK','Snapshot repetition and timing summaries; no patient rows returned.',
         tables={'sources':[tables['snapshots']], 'snapshots_per_patient':histogram.to_dict('records')},
         details={'distinct_date_spacing':spacing.to_dict('records'), 'positive_timing':positive_timing.to_dict('records'),
                  'positive_before_latest_is_not_automatically_invalid':True,
                  'spacing_uses_distinct_nonnull_dates':'Duplicate-key failure is reported separately by E04.'})

attempt('E04B_CALENDAR',evidence_snapshot_calendar)
attempt('E04B_MONTH_SPLIT',evidence_monthly_split_profile)
attempt('E04B_REPETITION',evidence_patient_snapshot_repetition)


## E05 — Recover the 1,028-feature map and saved tensor contract

**Why:** The feature map defines what `F0000…F1027` mean and their order. The saved tensor must have one row per patient-snapshot/month and agree with the frozen labels. This cell inspects the map and key columns only; it does not materialize patient-level feature values locally.

**Output:** The complete ordered feature map (paginated for sharing), map count, ordered-map hash, category-family counts, alias/schema checks, and aggregate tensor key checks. An exact count of 1,028 and a valid twelve-step grid prove structure only. The values `0…11` cannot prove which end of the sequence is newest, whether step zero was capped at END_DT, or whether counts mean claims, lines, diagnoses, or distinct events. Those remain explicit lineage questions.

**Genie help if needed:** “Read only: recover the executed builder of DL_POC_FEATURE_MAP and DL_POC_TENSOR_MONTHLY. Return the source notebook/version, FEATURE construction and inclusion rules, event grain/deduplication, date predicate including END_DT, and TIME_STEP formula. Distinguish executed code from proposed code. Do not infer chronology or cutoff from column names alone.”


In [ ]:
evidence_feature_map = None

def evidence_map_audit():
    global evidence_feature_map
    t = evidence_require_columns(tables['feature_map'], {'FEATURE_INDEX','FEATURE_NAME','FEATURE_COLUMN'})
    frame = run_query(f'SELECT FEATURE_INDEX, FEATURE_NAME, FEATURE_COLUMN FROM {t} ORDER BY FEATURE_INDEX', max_rows=2048)
    evidence_feature_map = frame.copy()
    indices = pd.to_numeric(frame['FEATURE_INDEX'], errors='coerce')
    names = frame['FEATURE_NAME'].tolist()
    aliases = frame['FEATURE_COLUMN'].tolist()
    tensor_columns = columns(tables['tensor_monthly'])
    checks = {
        'feature_count_1028': len(frame) == 1028,
        'index_exact_0_to_1027': indices.tolist() == list(range(1028)),
        'unique_nonnull_feature_names': frame['FEATURE_NAME'].notna().all() and not frame['FEATURE_NAME'].duplicated().any(),
        'unique_nonnull_aliases': frame['FEATURE_COLUMN'].notna().all() and not frame['FEATURE_COLUMN'].duplicated().any(),
        'aliases_exact_F0000_to_F1027': aliases == [f'F{i:04d}' for i in range(1028)],
        'all_aliases_exist_in_tensor': all(isinstance(x,str) and x.upper() in tensor_columns for x in aliases),
    }
    map_records = frame.to_dict('records')
    encoded = json.dumps(map_records, sort_keys=True, default=str, separators=(',',':')).encode('utf-8')
    family_counts = {}
    for name in names:
        family = str(name).split('_',1)[0]
        family = family if family in {'RX','DX','PX'} else 'OTHER_OR_UNRESOLVED'
        family_counts[family] = family_counts.get(family,0) + 1
    actual_alias_cols = sorted(c for c in tensor_columns if re.fullmatch(r'F\d{4}', c))
    extra_aliases = sorted(set(actual_alias_cols)-set(x for x in aliases if isinstance(x,str)))
    checks['documented_family_counts_exact'] = family_counts == {'RX':42,'DX':490,'PX':496}
    checks['no_extra_tensor_aliases'] = not extra_aliases
    emit('E05_FEATURE_MAP', 'PASS' if all(checks.values()) else 'FAIL',
         'Live ordered vocabulary and alias structure; clinical semantics require builder evidence.',
         tables={'sources':[tables['feature_map'], tables['tensor_monthly']], 'feature_map':map_records}, details={
         'checks': checks, 'feature_count': len(frame), 'map_sha256': hashlib.sha256(encoded).hexdigest(),
         'family_counts': family_counts, 'historically_reported_family_counts': {'RX':42,'DX':490,'PX':496},
         'tensor_alias_count': len(actual_alias_cols),
         'extra_tensor_aliases': extra_aliases,
         'full_mapping_retained_in_runtime_variable': 'evidence_feature_map',
         'source_rule_verification': 'UNRESOLVED: labels, feature names and family counts alone are insufficient'})

def evidence_tensor_keys():
    t = evidence_require_columns(tables['tensor_monthly'], {'PATIENT_ID','END_DT','RESP','TIME_STEP'})
    s = evidence_require_columns(tables['snapshots'], {'PATIENT_ID','END_DT','RESP'})
    row = run_query(f"""SELECT COUNT(*) AS N_ROWS,
      COALESCE(SUM(IFF(PATIENT_ID IS NULL OR END_DT IS NULL,1,0)),0) AS NULL_KEYS,
      COALESCE(SUM(IFF(TRIM(TO_VARCHAR(PATIENT_ID)) = '',1,0)),0) AS BLANK_PATIENT_KEYS,
      COALESCE(SUM(IFF(RESP IS NULL OR RESP NOT IN (0,1),1,0)),0) AS INVALID_LABEL_ROWS,
      COALESCE(SUM(IFF(TIME_STEP IS NULL OR TIME_STEP NOT IN (0,1,2,3,4,5,6,7,8,9,10,11),1,0)),0) AS INVALID_STEP_ROWS,
      COUNT(*) - (SELECT COUNT(*) FROM (SELECT PATIENT_ID, END_DT, TIME_STEP FROM {t} GROUP BY 1,2,3)) AS DUPLICATE_STEP_EXCESS
      FROM {t}""", max_rows=1).iloc[0].to_dict()
    keys = run_query(f"""WITH a AS (SELECT PATIENT_ID,END_DT, MIN(RESP) AS RESP, COUNT(*) AS N,
        COUNT(DISTINCT RESP) AS LABELS, SUM(IFF(RESP IS NULL,1,0)) AS NULL_LABELS FROM {s} GROUP BY 1,2),
      b AS (SELECT PATIENT_ID,END_DT,MIN(RESP) AS RESP,COUNT(*) AS N,
        COUNT(DISTINCT RESP) AS LABELS, SUM(IFF(RESP IS NULL,1,0)) AS NULL_LABELS,
        COUNT(DISTINCT TIME_STEP) AS STEPS FROM {t} GROUP BY 1,2)
      SELECT COALESCE(SUM(IFF(a.N IS NOT NULL AND b.N IS NULL,1,0)),0) AS MISSING_SNAPSHOTS,
        COALESCE(SUM(IFF(a.N IS NULL AND b.N IS NOT NULL,1,0)),0) AS EXTRA_SNAPSHOTS,
        COALESCE(SUM(IFF(a.N IS NOT NULL AND b.N IS NOT NULL AND a.RESP IS DISTINCT FROM b.RESP,1,0)),0) AS LABEL_MISMATCHES,
        COALESCE(SUM(IFF(b.N <> 12 OR b.STEPS <> 12,1,0)),0) AS INCOMPLETE_OR_DUPLICATE_SEQUENCES,
        COALESCE(SUM(IFF(a.N > 1,1,0)),0) AS DUPLICATE_FROZEN_KEYS,
        COALESCE(SUM(IFF(a.LABELS <> 1 OR a.NULL_LABELS > 0 OR b.LABELS <> 1 OR b.NULL_LABELS > 0,1,0)),0) AS INCONSISTENT_LABEL_KEYS
      FROM a FULL OUTER JOIN b ON a.PATIENT_ID=b.PATIENT_ID AND a.END_DT=b.END_DT""", max_rows=1).iloc[0].to_dict()
    ok = int(row['N_ROWS']) == 23151 * 12 and all(int(v) == 0 for k,v in row.items() if k != 'N_ROWS') and all(int(v) == 0 for v in keys.values())
    emit('E05_TENSOR_KEYS', 'PASS' if ok else 'FAIL',
         'Saved tensor row/key/label structure, without scanning all feature values.',
         tables=[tables['tensor_monthly'],tables['snapshots']], details={'row_audit':row,'key_audit':keys,
         'expected_rows':23151*12,'documented_orientation':'0=newest; not proven by key checks',
         'cutoff_verification':'UNRESOLVED','counting_grain_verification':'UNRESOLVED',
         'coverage_warning':'An all-zero row does not distinguish no events from unavailable observation.'})

attempt('E05_FEATURE_MAP', evidence_map_audit)
attempt('E05_TENSOR_KEYS', evidence_tensor_keys)


## E05B — Optional full tensor value audit (off by default)

**Why:** A full scan can identify missing, nonfinite, negative, nonintegral, and constant count channels. It reads all 1,028 value columns remotely in small batches and may be expensive. Run the default notebook first; enable `FULL_TENSOR_AUDIT` in configuration only when there is enough time/compute. Structural evidence from E05 remains useful without this scan.

**Output:** Aggregate feature-quality totals and at most 25 flagged feature names. No patient values are emitted. Null/nonfinite/negative/nonintegral counts fail the numeric count contract; constant features are reported for subsequent TRAIN-only selection rather than automatically removed. A clean result still cannot certify historical cutoff or mapping semantics.

**Genie help if needed:** “Estimate the read-only scan cost of the saved 277,812 by 1,028 tensor. If needed, run the same aggregate checks in the warehouse in batches. Do not sample and label the result a full audit, replace zero values, or write a new tensor.”


In [ ]:
evidence_tensor_feature_quality = None

def evidence_full_tensor_audit():
    global evidence_tensor_feature_quality
    if not FULL_TENSOR_AUDIT:
        emit('E05B_TENSOR_VALUES', 'SKIPPED', 'Optional full value scan is disabled; this is not a PASS.',
             details={'enable_with':'FULL_TENSOR_AUDIT = True','historical_cutoff_still_requires_source_evidence':True})
        return
    if evidence_feature_map is None or len(evidence_feature_map) != 1028:
        raise ValueError('A complete 1028-row feature map is required before the optional scan.')
    t = qi(tables['tensor_monthly'])
    records = evidence_feature_map.to_dict('records')
    quality = []
    for offset in range(0,len(records),32):
        batch = records[offset:offset+32]
        select = ['COUNT(*) AS N_ROWS']
        for j,r in enumerate(batch):
            alias = r['FEATURE_COLUMN']
            if not isinstance(alias,str) or not re.fullmatch(r'F\d{4}', alias):
                raise ValueError('Unexpected tensor alias; inspect the feature map.')
            c = qi(alias)
            select.extend([f'COUNT({c}) AS N{j}', f'COUNT_IF({c} < 0) AS NEG{j}',
                f'COUNT_IF({c} <> TRUNC({c})) AS FRAC{j}', f'COUNT_IF({c} <> 0) AS NZ{j}',
                f"COUNT_IF(UPPER(TO_VARCHAR({c})) IN ('NAN','INF','INFINITY','-INF','-INFINITY')) AS NONFINITE{j}",
                f'MIN({c}) AS MIN{j}', f'MAX({c}) AS MAX{j}'])
        row = run_query('SELECT '+', '.join(select)+' FROM '+t, max_rows=1).iloc[0]
        for j,r in enumerate(batch):
            def _n(key):
                value = row[key]
                return 0 if pd.isna(value) else int(value)
            quality.append({'FEATURE_COLUMN':r['FEATURE_COLUMN'],'FEATURE_NAME':r['FEATURE_NAME'],
                'ROWS':_n('N_ROWS'),'NULLS':_n('N_ROWS')-_n(f'N{j}'),
                'NEGATIVE':_n(f'NEG{j}'),'NONINTEGER':_n(f'FRAC{j}'),'NONFINITE':_n(f'NONFINITE{j}'),'NONZERO':_n(f'NZ{j}'),
                'CONSTANT': bool(pd.notna(row[f'MIN{j}']) and pd.notna(row[f'MAX{j}']) and row[f'MIN{j}']==row[f'MAX{j}'])})
    evidence_tensor_feature_quality = pd.DataFrame(quality)
    flags = [r for r in quality if r['NULLS'] or r['NEGATIVE'] or r['NONINTEGER'] or r['NONFINITE'] or r['CONSTANT']]
    totals = {key:sum(int(r[key]) for r in quality) for key in ['NULLS','NEGATIVE','NONINTEGER','NONFINITE','NONZERO']}
    emit('E05B_TENSOR_VALUES', 'FAIL' if any(totals[k] for k in ['NULLS','NEGATIVE','NONINTEGER','NONFINITE']) else 'PASS',
         'Full stored-count numeric audit; not a temporal lineage certification.', tables=[tables['tensor_monthly']],
         details={'features_scanned':len(quality),'totals':totals,'constant_features':sum(r['CONSTANT'] for r in quality),
                  'flagged_features_count':len(flags),'first_25_flags':flags[:25],
                  'full_aggregate_results_retained_in_runtime_variable':'evidence_tensor_feature_quality'})

attempt('E05B_TENSOR_VALUES', evidence_full_tensor_audit)


## E06 — Recover the exact 49-feature order and fixed encoding parameters

**Why:** The configured `MODEL_TYPE.FEATURES` order must agree with `FINAL_MODEL.SEQ` and the columns in MODEL_DATA. The existing notebook reads already-encoded V63 values; changing order or refitting a cap here would change the experiment. We retrieve the declared type/cap for each selected feature without applying transformations or ranking features.

**Output:** Ordered feature names, type/cap metadata, source hashes, and agreement checks. These verify the live configuration, **not** how or on which patients the parameters were historically fitted. The current notebook declares `upstream_fitting_uses_RESP=True`; that declaration is a reason to retrieve the generating procedure and fitting records, not sufficient proof of a specific label-dependent formula. Sharing the same uncertain preprocessing between two models does not remove leakage risk.

**Genie help if needed:** “Read only: explain how V63 FEATURES_SUMMARY.VALUE_P and VAR_TYP and the selected MODEL_TYPE.FEATURES were computed in the historical run. Return procedure/version and fitting population/split restrictions. Separate present-day procedure text from evidence of the historically executed version. Do not refit parameters or reorder selected features.”


In [ ]:
evidence_v63_features = None

def evidence_v63_configuration():
    global evidence_v63_features
    t = evidence_require_columns(tables['model_type'], {'MODEL_TYPE','FEATURES'})
    config = run_query(f'SELECT MODEL_TYPE, FEATURES FROM {t}', max_rows=10)
    if len(config) != 1:
        emit('E06_FEATURE_CONFIGURATION','FAIL','Expected exactly one frozen MODEL_TYPE row.',
             tables=[tables['model_type']],details={'actual_rows':len(config)})
        return
    value = config.iloc[0]['FEATURES']
    features = json.loads(value) if isinstance(value,str) else value
    if isinstance(features,np.ndarray):
        features = features.tolist()
    if not isinstance(features,list) or any(not isinstance(x,str) or not x or x != x.strip() for x in features):
        raise ValueError('FEATURES is not an ordered array of exact nonempty names.')
    evidence_v63_features = features
    model_columns = columns(tables['model_data'])
    prohibited = {'PATIENT_ID','END_DT','START_DT','RESP','SPLIT','RND','SCORE','DECILE','CENTILE','MILLILE'}
    checks = {'count_49':len(features)==49,'no_case_insensitive_duplicates':len({x.upper() for x in features})==len(features),
        'no_identifiers_targets_or_outputs':not bool(prohibited.intersection(x.upper() for x in features)),
        'all_predictors_in_MODEL_DATA':all(x.upper() in model_columns for x in features)}
    emit('E06_FEATURE_CONFIGURATION','PASS' if all(checks.values()) else 'FAIL',
         'Configured V63 feature list; MODEL_TYPE order is retained.',
         tables=[tables['model_type'],tables['model_data']],details={'checks':checks,
         'configured_model_type':str(config.iloc[0]['MODEL_TYPE']),
         'ordered_features':[{'order':i,'feature':x} for i,x in enumerate(features)],
         'ordered_feature_sha256':hashlib.sha256(json.dumps(features,separators=(',',':')).encode()).hexdigest()})

def evidence_v63_final_order():
    if evidence_v63_features is None:
        raise ValueError('Configured V63 feature order was not recovered.')
    t = evidence_require_columns(tables['final_model'], {'FEATURES','SEQ'})
    final = run_query(f'SELECT FEATURES, SEQ FROM {t} ORDER BY SEQ', max_rows=2048)
    rank = pd.to_numeric(final['SEQ'],errors='coerce')
    rank_valid = bool(rank.notna().all() and np.isfinite(rank).all() and rank.eq(np.floor(rank)).all())
    checks = {'all_names_present':final['FEATURES'].notna().all(),
              'names_unique':not final['FEATURES'].duplicated().any(),
              'ranks_finite_integer_unique':rank_valid and not rank.duplicated().any(),
              'exact_order_agreement':final['FEATURES'].tolist() == evidence_v63_features}
    emit('E06_FINAL_ORDER','PASS' if all(checks.values()) else 'FAIL',
         'MODEL_TYPE order versus FINAL_MODEL.SEQ.',tables=[tables['final_model'],tables['model_type']],
         details={'checks':checks,'final_feature_count':len(final),
         'configuration_only':sorted(set(evidence_v63_features)-set(final['FEATURES'].dropna())),
         'final_only':sorted(set(final['FEATURES'].dropna())-set(evidence_v63_features))})

def evidence_v63_parameters():
    if evidence_v63_features is None:
        raise ValueError('Configured V63 feature list was not recovered.')
    t = evidence_require_columns(tables['features_summary'], {'FEATURES','VAR_TYP','VALUE_P'})
    duplicates = run_query(f"""SELECT COUNT(*) AS DUPLICATED_FEATURE_NAMES FROM
        (SELECT UPPER(FEATURES) AS F FROM {t} GROUP BY 1 HAVING COUNT(*) > 1)""",max_rows=1)
    selected = ','.join(lit(x) for x in evidence_v63_features)
    frame = run_query(f'SELECT FEATURES, VAR_TYP, VALUE_P FROM {t} WHERE FEATURES IN ({selected})',max_rows=2048)
    missing = sorted(set(evidence_v63_features)-set(frame['FEATURES'].dropna()))
    caps = pd.to_numeric(frame['VALUE_P'],errors='coerce')
    types = frame['VAR_TYP'].astype('string').str.strip().str.upper()
    checks = {'no_duplicate_summary_feature_names':int(duplicates.iloc[0]['DUPLICATED_FEATURE_NAMES'])==0,
        'all_selected_features_have_parameters':not missing,
        'exactly_one_parameter_row_per_selected_feature':len(frame)==len(evidence_v63_features) and not frame['FEATURES'].duplicated().any(),
        'caps_finite_nonnegative':bool(caps.notna().all() and np.isfinite(caps).all() and caps.ge(0).all()),
        'types_declared_binary_or_numeric':bool(types.isin(['BINARY','NUMERIC']).all())}
    order = {x:i for i,x in enumerate(evidence_v63_features)}
    frame = frame.assign(FEATURE_ORDER=frame['FEATURES'].map(order)).sort_values('FEATURE_ORDER')
    records = frame.to_dict('records')
    emit('E06_FIXED_PARAMETERS','PASS' if all(checks.values()) else 'FAIL',
         'Live fixed parameter integrity only; fitting provenance remains separate.',
         tables=[tables['features_summary']],details={'checks':checks,'missing_selected_parameters':missing,
         'parameter_rows':records,
         'parameter_rows_sha256':hashlib.sha256(json.dumps(records,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest(),
         'upstream_fit_population':'UNVERIFIED',
         'upstream_RESP_use':'CURRENT_NOTEBOOK_DECLARATION_TRUE; exact historical formula/execution still requires evidence',
         'current_tensor_semantics':'One encoded snapshot vector repeated at valid monthly/quarterly positions; historical values are not reconstructed.'})

attempt('E06_FEATURE_CONFIGURATION', evidence_v63_configuration)
attempt('E06_FINAL_ORDER', evidence_v63_final_order)
attempt('E06_FIXED_PARAMETERS', evidence_v63_parameters)


## E07 — Dated source access, available schema, and remaining semantic questions

**Why:** We need the exact frozen claims vintage, enrollment intervals, mappings, and provider lookup to audit sequence construction. A readable table or a date column establishes availability today; it does not prove which table/version was used in the earlier run, when claims became available, or whether a mapping is historically valid. This cell probes access with no patient row output and returns column-name evidence for resolving those questions.

**Output:** One small schema/access report per source. Candidate dated claims FQNs are leads recovered from earlier proposed queries, not certified executed bindings. `READ_OK` means the table can be read. Source cutoff, event-grain, paid/reversed handling, mapping multiplicity and as-of semantics remain `UNRESOLVED` until supported by generating code or an empirical reconstruction.

**Genie help if needed:** “Read only: resolve the dated 20260825 medical/pharmacy tables and original V63 MAPPING, DX_CSSR_LATEST and narcolepsy basket bindings. Return exact FQNs and the original event ID, date, category, status/reversal and mapping-grain rules. Explain any missing permission. Do not substitute LATEST tables, invent missing dates or treat current lookup attributes as historical.”


In [ ]:
def evidence_source_access(key, role):
    fqn = tables.get(key)
    if not fqn:
        emit('E07_'+key.upper(),'UNAVAILABLE','Source binding is not yet established.',
             details={'role':role,'next_step':'Use the documented Genie prompt to resolve the exact FQN; do not guess.'})
        return
    names = sorted(columns(fqn))
    # The returned value is only 0/1; no source row or source value leaves the warehouse.
    readable = run_query(f'SELECT COUNT(*) AS READABLE_SAMPLE_ROWS FROM (SELECT 1 AS PRESENT FROM {qi(fqn)} LIMIT 1)',max_rows=1)
    categories = {
        'event_identifiers':[x for x in names if x in {'MEDICAL_EVENT_ID','PHARMACY_EVENT_ID','CLAIM_ID','CLAIM_LINE_ID','EVENT_ID'}],
        'event_or_interval_dates':[x for x in names if x in {'SERVICE_DATE','SERVICE_TO_DATE','FILL_DATE','START_DATE','END_DATE','START_DT','END_DT'}],
        'possible_availability_dates':[x for x in names if any(t in x for t in ['LOAD_DATE','LOAD_DT','RECEIVED','INGEST','ADJUDICAT'])],
        'status_or_reversal_fields':[x for x in names if any(t in x for t in ['TRANSACTION_RESULT','TRANSACTION_STATUS','REVERS','PAID','CLAIM_STATUS'])],
        'mapping_or_category_fields':[x for x in names if any(t in x for t in ['CODE','CATEGORY','DESCRIPTION','GENERIC','NDC','LEVEL','SPECIALTY'])],
    }
    emit('E07_'+key.upper(),'READ_OK','Read access and available schema verified; historical semantics are unresolved.',
         tables=[fqn],details={'role':role,'has_at_least_one_row':bool(int(readable.iloc[0]['READABLE_SAMPLE_ROWS'])),
         'column_count':len(names),'columns':names,'field_leads':categories,
         'historical_executed_binding':'UNRESOLVED','as_of_record_availability':'UNRESOLVED',
         'counting_deduplication_and_mapping_rules':'UNRESOLVED'})

for _source_key,_role in [
    ('enrollment','Frozen observation intervals used for padding validity'),
    ('medical_dated','Candidate frozen medical claims vintage 20260825'),
    ('pharmacy_dated','Candidate frozen pharmacy claims vintage 20260825'),
    ('mapping','Original V63 code/category mapping'),
    ('dx_mapping','Diagnosis category mapping; exact binding may be unresolved'),
    ('hcp_lookup','Current provider lookup; historical as-of validity is not established')]:
    attempt('E07_'+_source_key.upper(),lambda k=_source_key,r=_role: evidence_source_access(k,r))


## E08 — Current procedure evidence and historical provenance

**Why:** Recover the available V63 feature creation, encoding, fitting and scoring procedures without asking for information already in the warehouse. A current definition is evidence about current code; it does not establish which version or population fitted the historical model.

**Output meaning:** The table reports source hashes, availability, timestamps and structural indicators. A `RESP` reference or a percentile expression is a lead for review, not proof of leakage or proof of safe fitting. Source text stays in private notebook memory and is never printed. Missing permissions produce an explicit unknown.

**Genie help:** “Review `private_procedure_sources` in this work environment. Identify the exact V63 encoding/feature-selection formulas, whether RESP participates, which rows fitted their parameters, and event/label cutoffs. Cite procedure name, source hash and line numbers; do not output credentials or patient rows. Distinguish current definitions from the version actually executed historically.”

In [ ]:

import re
private_procedure_sources = {}
procedure_evidence = []
def collect_procedure_evidence():
    names = ('FEATURE_CREATION_PROD', 'FEATURE_ENGINEERING_PROD',
             'COMPARE_CLASSIFICATION_MODELS_PROD', 'TRAIN_CLASSIFICATION_PROD',
             'BACK_TESTING_PROD', 'SCORING_PROD')
    query = """SELECT PROCEDURE_SCHEMA, PROCEDURE_NAME, ARGUMENT_SIGNATURE,
               CREATED, LAST_ALTERED, PROCEDURE_DEFINITION
        FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.PROCEDURES
        WHERE PROCEDURE_SCHEMA IN ('DS_ML_DEV', 'DS_ML', 'DS_ML_PRD')
          AND PROCEDURE_NAME IN (""" + ','.join(lit(n) for n in names) + ')'
    frame = run_query(query, max_rows=100)
    patterns = {
        'RESP_referenced': r'\bRESP\b',
        'VALUE_P_referenced': r'\bVALUE_P\b',
        'VAR_TYP_referenced': r'\bVAR_TYP\b',
        'percentile_expression_present': r'\bPERCENTILE_(?:CONT|DISC)\b|\bQUANTILE\b',
        'fit_expression_present': r'\.FIT\s*\(|\bTRAIN_CLASSIFICATION',
        'split_expression_present': r'\bTRAIN_TEST_SPLIT\b|\bSTRATIF\w*|\bSPLIT\b',
        'event_cutoff_terms_present': r'\bEND_DT\b|\bSERVICE_DATE\b|\bFILL_DATE\b',
        'label_horizon_terms_present': r'\bDATEADD\b|\bDATEDIFF\b|\b90\b',
    }
    for row in frame.to_dict('records'):
        body = row.get('PROCEDURE_DEFINITION')
        available = isinstance(body, str) and bool(body.strip())
        signature = str(row.get('ARGUMENT_SIGNATURE') or '')
        key = '.'.join(['DSVC_TAKEDA_TA_PRIVATE', str(row['PROCEDURE_SCHEMA']), str(row['PROCEDURE_NAME'])])
        signature_hash = hashlib.sha256(signature.encode()).hexdigest()
        if available:
            private_procedure_sources[(key, signature_hash)] = body
        record = {'procedure': key, 'signature_sha256': signature_hash,
                  'definition_available': available,
                  'definition_sha256': hashlib.sha256(body.encode()).hexdigest() if available else None,
                  'created': str(row.get('CREATED')), 'last_altered': str(row.get('LAST_ALTERED')),
                  'historical_executed_version': 'UNKNOWN', 'historical_fit_membership': 'UNKNOWN'}
        for label, pattern in patterns.items():
            record[label] = bool(re.search(pattern, body, flags=re.I)) if available else None
        procedure_evidence.append(record)
    emit('E08', 'READ_OK' if procedure_evidence else 'UNAVAILABLE',
         'Current source metadata only; semantic review and historical execution provenance remain separate.',
         tables=procedure_evidence,
         details={'definitions_kept_private': len(private_procedure_sources), 'raw_source_printed': False,
                  'missing_procedure_names': sorted(set(names)-set(frame.get('PROCEDURE_NAME', pd.Series(dtype=str))))})
attempt('E08', collect_procedure_evidence)


## E09 — Recover existing V63 benchmark evidence and check comparable populations

**Why:** The LightGBM benchmark must come from its actual saved metrics. Its scored tables also tell us whether patient-snapshot keys and labels match the Transformer cohort. This cell does not fit a model, score patients, select a threshold, or calculate new TEST performance.

**Output meaning:** Saved metrics retain their original column names and definitions remain unverified until reviewed. Recognized candidate score columns are checked for missing, unparseable and nonfinite values plus finite ranges; values outside [0,1] can be valid logits and do not establish an error. Score semantics remain unverified. Exact cohort coverage and label agreement are necessary, but cannot prove that benchmark patients were excluded from LightGBM fitting. A model-level metrics table that unexpectedly contains patient identifiers is not returned.

**Genie help:** “Using the reported benchmark schemas and V63 training procedure evidence, identify the recorded AP/PR-AUC definition, split/population, threshold rule and model fitting membership. If a field is absent, mark it unknown. Check the documented private stage model `@DSVC_TAKEDA_TA_PRIVATE.DS_ML_PRD.PYTHON_UDF/TAK861_Tx_Ready_V63_lightgbm.joblib` using metadata only; do not deserialize the model or publish its binary.”

In [ ]:

benchmark_evidence = {}
BENCHMARK_NUMERIC_FIELDS = {
    'AP', 'AVERAGE_PRECISION', 'PR_AUC', 'PRAUC', 'AUPRC', 'ROC_AUC', 'ROC_AUC_SCORE', 'AUC',
    'PRECISION', 'RECALL', 'F1', 'F1_SCORE', 'ACCURACY', 'BALANCED_ACCURACY',
    'LOG_LOSS', 'BRIER_SCORE', 'LIFT', 'LIFT_AT_10', 'LIFT_AT_10_PERCENT', 'TOP10_LIFT',
    'RECALL_AT_10', 'RECALL_AT_10_PERCENT', 'TOP10_RECALL', 'TOP10_PRECISION',
    'THRESHOLD', 'TP', 'TN', 'FP', 'FN', 'SNAPSHOTS', 'PATIENTS', 'N', 'COUNT',
    'POSITIVES', 'NEGATIVES', 'PREVALENCE', 'N_FEATURES', 'FEATURE_COUNT', 'RANK', 'VALUE',
}
BENCHMARK_LABEL_FIELDS = {'MODEL', 'MODEL_NAME', 'MODEL_TYPE', 'ALGORITHM', 'CLASSIFIER', 'SPLIT', 'DATASET', 'METRIC'}
def known_table(key):
    value = tables.get(key)
    if not isinstance(value, str):
        raise ValueError('Required configured table key is unavailable: ' + key)
    return value

def collect_existing_benchmark_metrics():
    table = known_table('model_comparison')
    cols = columns(table)
    if {'PATIENT_ID', 'END_DT', 'RESP'} & cols:
        emit('E09.metrics', 'INCONCLUSIVE', 'Table contains patient-level column names; no rows returned.',
             details={'source': table, 'columns': sorted(cols)})
        return
    selected = sorted(cols & (BENCHMARK_NUMERIC_FIELDS | BENCHMARK_LABEL_FIELDS))
    # Also recognize explicit TRAIN_/VALIDATION_/TEST_ metric columns without computing them.
    selected += sorted(c for c in cols-set(selected) if any(c == prefix+m for prefix in
                       ('TRAIN_', 'TRAINING_', 'VALIDATION_', 'VALID_', 'TEST_') for m in BENCHMARK_NUMERIC_FIELDS))
    if not selected:
        emit('E09.metrics', 'INCONCLUSIVE', 'No reviewed aggregate metric columns recognized; request a schema-specific reader.',
             details={'source': table, 'columns': sorted(cols)})
        return
    frame = run_query('SELECT '+','.join(qi(c) for c in selected)+' FROM '+qi(table), max_rows=100)
    safe = []
    omitted_values = 0
    for row in frame.to_dict('records'):
        item = {}
        for key, value in row.items():
            if pd.isna(value):
                item[key] = None
            elif key in BENCHMARK_LABEL_FIELDS:
                label = str(value)
                if re.fullmatch(r'[A-Za-z][A-Za-z0-9_ .%()/+-]{0,99}', label):
                    item[key] = label
                else:
                    omitted_values += 1
            else:
                number = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
                if pd.notna(number) and np.isfinite(float(number)):
                    item[key] = float(number)
                else:
                    omitted_values += 1
        if item:
            safe.append(item)
    benchmark_evidence['existing_metrics'] = safe
    emit('E09.metrics', 'READ_OK' if safe else 'UNAVAILABLE',
         'Previously saved aggregate values only; no new performance calculation.', tables=safe,
         details={'source': table, 'unrecognized_columns': sorted(cols-set(selected)),
                  'omitted_unrecognized_values': omitted_values,
                  'metric_definitions': 'UNVERIFIED', 'fitting_membership': 'UNKNOWN',
                  'ap_and_trapezoidal_pr_auc_are_not_interchangeable': True})


RECOGNIZED_SCORE_COLUMNS = {
    'SCORE','P_RESP1','P_RESP','PROBABILITY','PROBABILITY_1','PRED_PROBA',
    'PREDICTION_PROBABILITY','PREDICTED_PROBABILITY','RESPONSE_PROBABILITY','PROB_RESP','PREDICTED_SCORE',
}

def collect_score_quality(table, key, cols):
    candidates=sorted(cols & RECOGNIZED_SCORE_COLUMNS)
    schema_rows=globals().get('schemas',{}).get(table,[])
    types={str(row.get('COLUMN_NAME','')).upper():str(row.get('DATA_TYPE','UNKNOWN')) for row in schema_rows}
    if not candidates:
        emit('E09.score_quality.'+key,'INCONCLUSIVE','No explicitly recognized candidate score column; inspect the reported schema.',
             details={'source':table,'score_semantics':'UNVERIFIED'})
        return
    for score_column in candidates:
        def inspect(score_column=score_column):
            # TRY_TO_DOUBLE handles string-backed candidates without assuming their semantics.
            # Snowflake renders special floating-point values as NaN / inf / -inf.
            query=f"""WITH candidate AS (
              SELECT {qi(score_column)} IS NULL AS RAW_MISSING,
                     TRY_TO_DOUBLE(TO_VARCHAR({qi(score_column)})) AS V
              FROM {qi(table)}
            ), quality AS (
              SELECT RAW_MISSING,V,
                     UPPER(TO_VARCHAR(V)) IN ('NAN','INF','-INF','INFINITY','-INFINITY') AS NONFINITE,
                     CASE WHEN V IS NOT NULL AND
                          UPPER(TO_VARCHAR(V)) NOT IN ('NAN','INF','-INF','INFINITY','-INFINITY')
                          THEN V ELSE NULL END AS FINITE_VALUE
              FROM candidate
            )
            SELECT COUNT(*) AS N_ROWS,
                   COUNT_IF(RAW_MISSING) AS NULL_SCORE_VALUES,
                   COUNT_IF(NOT RAW_MISSING AND V IS NULL) AS UNPARSEABLE_SCORE_VALUES,
                   COUNT_IF(NONFINITE) AS NONFINITE_SCORE_VALUES,
                   COUNT(FINITE_VALUE) AS FINITE_SCORE_VALUES,
                   MIN(FINITE_VALUE) AS MIN_FINITE_SCORE,MAX(FINITE_VALUE) AS MAX_FINITE_SCORE,
                   COUNT_IF(FINITE_VALUE<0 OR FINITE_VALUE>1) AS FINITE_VALUES_OUTSIDE_0_1
            FROM quality"""
            quality=run_query(query,max_rows=1).to_dict('records')
            emit('E09.score_quality.'+key+'.'+score_column,'READ_OK',
                 'Candidate score quality only; values outside [0,1] may be valid logits. No discrimination or threshold metrics calculated.',
                 tables=quality,details={'source':table,'candidate_score_column':score_column,
                    'catalog_data_type':types.get(score_column,'UNKNOWN'),'score_semantics':'UNVERIFIED',
                    'conversion':'TRY_TO_DOUBLE for aggregate diagnostics only','new_test_metrics_calculated':False})
        attempt('E09.score_quality.'+key+'.'+score_column,inspect)

def collect_score_table_evidence(key):
    table = known_table(key)
    cols = columns(table)
    emit('E09.schema.'+key, 'READ_OK', 'Scored-table schema only.',
         details={'source': table, 'columns': sorted(cols)})
    collect_score_quality(table,key,cols)
    if not {'PATIENT_ID', 'END_DT', 'RESP'} <= cols:
        emit('E09.cohort.'+key, 'INCONCLUSIVE', 'Exact patient-snapshot/label comparison requires confirmed PATIENT_ID, END_DT and RESP columns.',
             details={'source': table})
        return
    query = f"""SELECT COUNT(*) AS N_ROWS, COUNT(DISTINCT PATIENT_ID) AS PATIENTS,
      COUNT(DISTINCT PATIENT_ID,END_DT) AS SNAPSHOT_KEYS,
      COUNT_IF(NULLIF(TRIM(TO_VARCHAR(PATIENT_ID)), '') IS NULL OR END_DT IS NULL) AS INVALID_KEYS,
      COUNT_IF(RESP=1) AS POSITIVES,
      COUNT_IF(RESP IS NULL OR RESP NOT IN (0,1)) AS INVALID_LABELS,
      MIN(END_DT) AS MIN_END_DT, MAX(END_DT) AS MAX_END_DT
      FROM {qi(table)}"""
    summary = run_query(query, max_rows=1).to_dict('records')
    split_table = known_table('frozen_split')
    if not {'PATIENT_ID', 'END_DT', 'RESP', 'SPLIT'} <= columns(split_table):
        emit('E09.cohort.'+key, 'INCONCLUSIVE', 'Scored cohort aggregates recovered; frozen split schema insufficient.',
             tables=summary, details={'source':table})
        return
    recognized_scores=sorted(cols & RECOGNIZED_SCORE_COLUMNS)
    alignment_score='SCORE' if 'SCORE' in recognized_scores else recognized_scores[0] if recognized_scores else None
    nonnull_score_sql='COUNT('+qi(alignment_score)+')' if alignment_score else 'CAST(NULL AS NUMBER)'
    query = f"""WITH score_keys AS (
       SELECT PATIENT_ID,END_DT,COUNT(*) AS SOURCE_ROWS,COUNT(RESP) AS NONNULL_LABELS,
              {nonnull_score_sql} AS NONNULL_SCORES,
              MIN(RESP) AS MIN_RESP,MAX(RESP) AS MAX_RESP
       FROM {qi(table)} GROUP BY PATIENT_ID,END_DT)
       SELECT f.SPLIT,COUNT(*) AS EXPECTED_SNAPSHOTS,
              COUNT(DISTINCT CASE WHEN s.SOURCE_ROWS IS NOT NULL THEN f.PATIENT_ID END) AS MATCHED_PATIENTS,
              COUNT_IF(s.NONNULL_SCORES>0) AS MATCHES_WITH_ANY_NONNULL_SCORE,
              COUNT_IF(s.SOURCE_ROWS IS NOT NULL AND s.NONNULL_SCORES=s.SOURCE_ROWS) AS MATCHES_WITH_ALL_NONNULL_SCORES,
              COUNT_IF(s.SOURCE_ROWS IS NOT NULL) AS MATCHED_SNAPSHOTS,
              COUNT_IF(s.SOURCE_ROWS IS NULL) AS MISSING_SNAPSHOTS,
              COUNT_IF(s.SOURCE_ROWS>1) AS NONUNIQUE_MATCHES,
              COUNT_IF(s.SOURCE_ROWS IS NOT NULL AND
                       (f.RESP IS NULL OR f.RESP NOT IN (0,1) OR s.NONNULL_LABELS<>s.SOURCE_ROWS OR s.MIN_RESP<>f.RESP OR s.MAX_RESP<>f.RESP)) AS LABEL_PROBLEMS
       FROM {qi(split_table)} f LEFT JOIN score_keys s
            ON s.PATIENT_ID=f.PATIENT_ID AND s.END_DT=f.END_DT
       GROUP BY f.SPLIT ORDER BY f.SPLIT"""
    alignment = run_query(query, max_rows=10).to_dict('records')
    extra = run_query(f"""WITH s AS (SELECT DISTINCT PATIENT_ID,END_DT FROM {qi(table)})
       SELECT COUNT(*) AS EXTRA_SNAPSHOT_KEYS FROM s
       WHERE NOT EXISTS (SELECT 1 FROM {qi(split_table)} f
                         WHERE f.PATIENT_ID=s.PATIENT_ID AND f.END_DT=s.END_DT)""", max_rows=1).to_dict('records')
    benchmark_evidence[key] = {'cohort':summary,'alignment':alignment,'extras':extra}
    emit('E09.cohort.'+key, 'READ_OK', 'Key/label comparability evidence; fitting exclusion remains UNKNOWN.',
         tables=alignment, details={'source':table, 'cohort':summary, 'extra_keys':extra,
                                   'fair_comparison_established': False, 'alignment_candidate_score_column':alignment_score,
                                   'score_semantics':'UNVERIFIED', 'new_test_metrics_calculated':False})
attempt('E09.metrics', collect_existing_benchmark_metrics)
for score_key in ('universe_scored', 'backtest_scored', 'patients_scored'):
    attempt('E09.'+score_key, lambda k=score_key: collect_score_table_evidence(k))


## E10 — Recover saved history with integrity checks and a strict safe-field reader

**Why:** Historical model summaries, training curves and evaluation reports are already stored in chunked warehouse artifact tables. Recover them from their actual names rather than reconstructing performance from memory or photographs.

**Output meaning:** First, inventory the artifact names and verify chunk metadata. Only explicitly allowed aggregate text reports and the restricted prepared manifest are fetched. Manifest fields are saved declarations, not independent proof of historical execution or fitting provenance. Feature lists become counts and order hashes; patient arrays, full per-feature audits and free source text are omitted. Their bytes, SHA-256 hash and chunk sequence must match before any result is extracted. The reader returns a strict allowlist of metric/configuration fields, omitting patient records, predictions, arbitrary strings and unfamiliar structures. Missing fields stay unavailable. Table creation/alteration timestamps are catalog evidence and are never treated as run dates. A report's intrinsic timestamp is recorded as a reported timestamp, not independent proof.

**Bounds:** At most `MAX_ARTIFACT_TABLES` historical tables plus `MAX_PREPARED_TABLES` prepared-input tables (default 4) and `MAX_ARTIFACT_BYTES` bytes per allowed report are read. Oversized or unfamiliar artifacts are inventoried but skipped. If tables are omitted, increase the cap deliberately and rerun after reviewing the output; no model execution is involved.

**Genie help:** “For skipped reports, inspect their schema privately and propose an aggregate-only field mapping. Do not return patient IDs, patient-level predictions, split assignments, tensors or model binaries. Find intrinsic run timestamps, input/feature/split hashes, metric definitions and historical test-use statements. Absence of a test-use statement does not prove TEST was untouched.”

In [ ]:

SAFE_REPORT_NAMES = {
    'training_summary.json', 'training_history.csv', 'summary.json', 'history.json',
    'evaluation_metadata.json', 'evaluation.json', 'global_metrics.csv', 'deciles.csv',
    'topk.csv', 'ties.csv', 'training_topk.csv', 'validation_topk.csv',
    'selection.json', 'comparison.csv', 'manifest.json',
}
ARTIFACT_COLUMNS = {'ARTIFACT_NAME','CHUNK_INDEX','CHUNK_COUNT','BYTE_LENGTH','SHA256','PAYLOAD_BASE64'}
SAFE_NUMERIC_KEYS = {
    'epoch','best_epoch','epochs_completed','early_stopping_epoch','epochs','patience','min_delta',
    'seed','batch_size','learning_rate','weight_decay','l1_lambda','l2_lambda','grad_clip',
    'd_model','n_heads','encoder_layers','feedforward_dim','dropout','input_dim','seq_len',
    'n_features','feature_count','selected_count','model_parameter_count','parameter_count',
    'train_pos_weight','pos_weight','threshold','validation_threshold','best_validation_average_precision',
    'average_precision','ap','pr_auc','roc_auc','auc','precision','recall','f1','f1_score','accuracy',
    'balanced_accuracy','log_loss','brier_score','prevalence','positive_rate','positives','negatives',
    'snapshots','patients','n','count','tp','tn','fp','fn','top10_lift','top10_recall','top10_precision',
    'lift','recall_at_10','lift_at_10','recall_at_10_percent','lift_at_10_percent','fraction',
    'top_fraction','top_percent','top_pct','k','decile','selected','selected_positives','captured_positives',
    'n_snapshots','n_resp1','n_selected','population_fraction','response_rate','decile_precision',
    'share_of_all_resp1','cumulative_n_snapshots','cumulative_population_fraction','cumulative_resp1',
    'cumulative_recall','cumulative_precision','decile_lift','cumulative_lift','overall_test_response_rate',
    'top_k_pct','actual_population_fraction','cutoff_score','n_tied_at_cutoff','n_tied_selected',
    'test_snapshots','test_positive_snapshots','optimization_loss','optimization_bce','optimization_l1','optimization_l2',
    'training_loss','validation_loss','training_average_precision','validation_average_precision',
    'training_top10_lift','validation_top10_lift','validation_roc_auc','training_ap','validation_ap',
    'test_ap','test_roc_auc','ap_gap','generalization_gap','all_padded_snapshots',
}
SAFE_BOOL_KEYS = {'training_complete','test_inference_performed','test_scored','test_retrospective',
                  'causal_masking_effect_established','synthetic_only','client_data_used','test_used_for_selection','tie_crosses_boundary'}
SAFE_ENUM_KEYS = {
    'split':{'train','training','validation','valid','test','train_validation','train_validation_test'},
    'representation':{'monthly','quarterly'},
    'model':{'monthly','quarterly','lightgbm','lgbmclassifier','transformer','logistic','logisticregression'},
    'model_kind':{'transformer','logistic','lightgbm'},
    'evaluation_scope':{'train_validation','train_validation_test'},
    'feature_mode':{'all','reduced','core'}, 'mode':{'all','reduced','core'},
    'resolved_device':{'cpu','cuda','cuda:0','cuda:1','mps'},
    'regularization_scope':{'linear_and_attention_weights'},
    'time_orientation':{'0=most_recent; increasing_index=older'},
    'mask_convention':{'valid=true means observed; src_key_padding_mask=~valid; masked mean pooling'},
    'stop_reason':{'validation_ap_patience','max_epochs','early_stopping','completed'},
}
SAFE_TOKEN_KEYS = {'run_id','evaluation_id','dataset_id','suite_id','tuning_id','candidate_id'}
SAFE_HASH_KEYS = {'manifest_sha256','feature_manifest_sha256','implementation_sha256',
                 'population_sha256','checkpoint_sha256','feature_names_sha256','feature_order_sha256',
                 'model_input_sha256','snapshot_manifest_sha256','split_config_sha256','x_sha256',
                 'availability_sha256','consumer_implementation_sha256','frozen_split_assignments_sha256',
                 'mask_sha256','parameter_sha256','preprocessing_sha256','valid_sha256','split_sha256','experiment_signature','prepared_manifest_sha256'}
SAFE_TIME_KEYS = {'completed_at_utc','started_at_utc','created_at_utc','run_timestamp','completed_at'}
SAFE_CONTAINERS = {
    'training_metrics','train_metrics','validation_metrics','test_metrics','metrics','comparison','gaps',
    'history','models','configuration','config','model_settings','model_config','training_settings','class_counts','runtime','input_hashes','contract',
    'training_topk','validation_topk','topk','deciles','ties','results','runs','candidates','selection',
    'monthly','quarterly','train','training','validation','valid','test','feature_manifest',
}

def metric_key_allowed(key):
    if key in SAFE_NUMERIC_KEYS:
        return True
    return any(key.startswith(prefix) and key[len(prefix):] in SAFE_NUMERIC_KEYS
               for prefix in ('train_','training_','validation_','test_'))

def safe_scalar(key, value):
    normalized = str(key).lower()
    if value is None:
        return (True, None) if metric_key_allowed(normalized) or normalized in SAFE_BOOL_KEYS else (False, None)
    if normalized in SAFE_BOOL_KEYS and isinstance(value, (bool, np.bool_)):
        return True, bool(value)
    if metric_key_allowed(normalized) and not isinstance(value, (bool, dict, list, tuple)):
        number = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
        if pd.notna(number) and np.isfinite(float(number)):
            return True, int(number) if isinstance(value, (int, np.integer)) else float(number)
    if normalized in SAFE_ENUM_KEYS and isinstance(value,str) and value.lower() in SAFE_ENUM_KEYS[normalized]:
        return True, value
    if normalized in SAFE_TOKEN_KEYS and isinstance(value,str) and re.fullmatch(r'[A-Z][A-Z0-9_]{0,100}',value):
        return True, value
    if normalized in SAFE_HASH_KEYS and isinstance(value,str) and re.fullmatch(r'[0-9a-fA-F]{64}',value):
        return True, value.lower()
    if normalized in SAFE_TIME_KEYS and isinstance(value,str) and re.fullmatch(r'\d{4}-\d\d-\d\d[T ][0-9:.+-]+Z?',value):
        return True, value
    return False, None


MANIFEST_ENUM_FIELDS = {
    'feature_policy': {'v63_snapshot_only_valid_timestep_broadcast_v1'},
    'padding_contract': {'independent_timestep_availability_v1'},
    'orientation': {'0=newest; snapshot-relative month buckets; newest ends at END_DT'},
    'preprocessing_policy': {'one_vector_per_eligible_train_snapshot_shared_v1'},
    'quarterly_mask_rule': {'OR of matching monthly triples'},
    'quarterly_feature_aggregation': {'None: same snapshot values; only enrollment masks change by bucket width.'},
}
MANIFEST_HASH_FIELDS = {'implementation_sha256','population_sha256','snapshot_feature_sha256',
                        'frozen_split_assignments_sha256'}
MANIFEST_SOURCE_KEYS = {'snapshots','model_data','model_type','final_model','features_summary','enrollment','frozen_split'}

def extract_safe_manifest(blob):
    manifest=json.loads(blob.decode('utf-8'))
    if not isinstance(manifest,dict):
        raise ValueError('Prepared manifest must be an object.')
    output=[]
    def add(field,value):
        output.append({'field':field,'value':value})
    def nonnegative_integer(value):
        return isinstance(value,int) and not isinstance(value,bool) and value>=0
    add('evidence_kind','SAVED_CONTRACT_DECLARATION_NOT_INDEPENDENT_VALIDATION')
    add('historical_execution_and_fit_provenance_verified',False)
    if nonnegative_integer(manifest.get('schema')):
        add('schema',manifest['schema'])
    dataset=manifest.get('dataset_id')
    if isinstance(dataset,str) and re.fullmatch(r'[A-Z][A-Z0-9_]{0,100}',dataset):
        add('dataset_id',dataset)
    vintage=manifest.get('source_vintage')
    if isinstance(vintage,str) and re.fullmatch(r'\d{8}',vintage):
        add('source_vintage_declared',vintage)
    for field in ('source_features','features','excluded_features'):
        items=manifest.get(field)
        if isinstance(items,list) and len(items)<=10000 and all(isinstance(v,str) and len(v)<=500 for v in items):
            add(field+'_count_declared',len(items))
            canonical=json.dumps(items,ensure_ascii=False,separators=(',',':')).encode('utf-8')
            add(field+'_ordered_sha256',hashlib.sha256(canonical).hexdigest())
    for field,allowed in MANIFEST_ENUM_FIELDS.items():
        value=manifest.get(field)
        if isinstance(value,str) and value in allowed:
            add(field+'_declared',value)
        elif isinstance(value,str):
            add(field+'_recognized',False)
            add(field+'_unrecognized_text_sha256',hashlib.sha256(value.encode()).hexdigest())
    for field in MANIFEST_HASH_FIELDS:
        value=manifest.get(field)
        if isinstance(value,str) and re.fullmatch(r'[0-9a-fA-F]{64}',value):
            add(field,value.lower())
    for field in ('encoding_applied_here',):
        if isinstance(manifest.get(field),bool):
            add(field+'_declared',manifest[field])
    source_tables=manifest.get('source_tables',{})
    if isinstance(source_tables,dict):
        for key in sorted(MANIFEST_SOURCE_KEYS):
            value=source_tables.get(key)
            if isinstance(value,str) and re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*',value):
                add('source_tables.'+key+'_declared',value)
    encoding=manifest.get('business_encoding',{})
    if isinstance(encoding,dict):
        for field in ('encoding_applied_here','snapshot_MODEL_DATA_already_encoded',
                      'caps_or_types_refitted_here','upstream_fitting_uses_RESP'):
            value=encoding.get(field)
            if isinstance(value,bool):
                add('business_encoding.'+field+'_declared',value)
        value=encoding.get('parameter_sha256')
        if isinstance(value,str) and re.fullmatch(r'[0-9a-fA-F]{64}',value):
            add('business_encoding.parameter_sha256',value.lower())
    representations=manifest.get('representations',{})
    if isinstance(representations,dict):
        for name in ('MONTHLY','QUARTERLY'):
            representation=representations.get(name)
            if not isinstance(representation,dict):
                continue
            prefix='representations.'+name+'.'
            shape=representation.get('shape')
            if isinstance(shape,list) and len(shape)==3 and all(nonnegative_integer(v) for v in shape):
                for axis,value in zip(('snapshots','timesteps','features'),shape):
                    add(prefix+'shape_'+axis+'_declared',value)
            if representation.get('value_space') in ('V63 encoded','raw counts','log1p counts'):
                add(prefix+'value_space_declared',representation['value_space'])
            for field in ('X_sha256','valid_sha256','availability_sha256'):
                value=representation.get(field)
                if isinstance(value,str) and re.fullmatch(r'[0-9a-fA-F]{64}',value):
                    add(prefix+field,value.lower())
            audit=representation.get('coverage_audit',{})
            if isinstance(audit,dict):
                for field in ('available_timesteps','unavailable_timesteps','unknown_timesteps','all_padded_snapshots'):
                    value=audit.get(field)
                    if nonnegative_integer(value):
                        add(prefix+'coverage_audit.'+field+'_declared',value)
                for field in ('full_period_coverage_required','closed_claims_filter_applied'):
                    if isinstance(audit.get(field),bool):
                        add(prefix+'coverage_audit.'+field+'_declared',audit[field])
    # Summarize known source categories without exposing the per-feature audit/provenance.
    audit=manifest.get('audit')
    if isinstance(audit,list):
        for value_source in ('V63_SNAPSHOT','TEMPORAL','APPROXIMATED','SNAPSHOT_ONLY'):
            count=sum(isinstance(row,dict) and row.get('VALUE_SOURCE')==value_source for row in audit)
            add('feature_audit.'+value_source+'_count_declared',count)
    add('patient_arrays_and_free_text_exported',False)
    handled={'schema','dataset_id','source_vintage','source_features','features','excluded_features',
             'source_tables','business_encoding','representations','encoding_applied_here','audit'}
    handled |= set(MANIFEST_ENUM_FIELDS) | MANIFEST_HASH_FIELDS
    return output,len(set(manifest)-handled)

def extract_safe_report(blob, name, max_records=400):
    # Deny arbitrary nested field names and free text even within allowlisted filenames.
    if name == 'manifest.json':
        return extract_safe_manifest(blob)
    if name.endswith('.csv'):
        source = pd.read_csv(io.BytesIO(blob)).to_dict('records')
    else:
        source = json.loads(blob.decode('utf-8'))
    output, omitted = [], 0
    def walk(value, path=(), depth=0):
        nonlocal omitted
        if depth > 8:
            omitted += 1
            return
        if isinstance(value,list):
            # Only traverse lists of aggregate objects, never raw vectors/records with identifiers.
            for i,item in enumerate(value):
                if isinstance(item,dict):
                    walk(item,path+(str(i),),depth+1)
                else:
                    omitted += 1
        elif isinstance(value,dict):
            lowered = {str(k).lower() for k in value}
            if lowered & {'patient_id','patient_ids','end_dt','prediction','predictions','score','scores','y_true','y_pred'}:
                omitted += len(value)
                return
            for key,item in value.items():
                k=str(key).lower()
                accepted, scalar = safe_scalar(k,item)
                if accepted:
                    if len(output) < max_records:
                        output.append({'field':'.'.join(path+(str(key),)), 'value':scalar})
                    else:
                        omitted += 1
                elif k in SAFE_CONTAINERS and isinstance(item,(list,dict)):
                    walk(item,path+(str(key),),depth+1)
                else:
                    omitted += 1
        else:
            omitted += 1
    walk(source)
    return output, omitted

def decode_checked_artifact(frame, maximum_bytes):
    if frame.empty:
        raise ValueError('No artifact chunks returned.')
    rows = frame.sort_values('CHUNK_INDEX').to_dict('records')
    def integer(v):
        if isinstance(v,(bool,np.bool_)) or not np.isfinite(float(v)) or int(v) != float(v):
            raise ValueError('Nonintegral chunk metadata.')
        return int(v)
    count=integer(rows[0]['CHUNK_COUNT']); size=integer(rows[0]['BYTE_LENGTH'])
    digest=str(rows[0]['SHA256']).lower()
    if not 1 <= count <= 10000 or not 0 <= size <= maximum_bytes or not re.fullmatch(r'[0-9a-f]{64}',digest):
        raise ValueError('Invalid or oversized artifact metadata.')
    if len(rows)!=count or [integer(r['CHUNK_INDEX']) for r in rows] != list(range(count)):
        raise ValueError('Missing or duplicate artifact chunks.')
    if any(integer(r['CHUNK_COUNT'])!=count or integer(r['BYTE_LENGTH'])!=size or str(r['SHA256']).lower()!=digest for r in rows):
        raise ValueError('Inconsistent artifact metadata.')
    pieces=[]; encoded_size=0
    for row in rows:
        piece=row['PAYLOAD_BASE64']
        if not isinstance(piece,str):
            raise ValueError('Non-text payload encoding.')
        encoded_size += len(piece)
        if encoded_size > 4*((maximum_bytes+2)//3):
            raise ValueError('Encoded payload exceeds byte bound.')
        pieces.append(piece)
    blob=base64.b64decode(''.join(pieces),validate=True)
    if len(blob)!=size or hashlib.sha256(blob).hexdigest()!=digest:
        raise ValueError('Artifact length or SHA-256 verification failed.')
    return blob

artifact_inventory = []
recovered_safe_reports = []
def collect_one_artifact_table(table, catalog_record):
    if not ARTIFACT_COLUMNS <= columns(table):
        return
    inventory=run_query(f"""SELECT ARTIFACT_NAME,MIN(BYTE_LENGTH) AS MIN_BYTES,MAX(BYTE_LENGTH) AS MAX_BYTES,
      MIN(CHUNK_COUNT) AS MIN_CHUNKS,MAX(CHUNK_COUNT) AS MAX_CHUNKS,COUNT(*) AS STORED_CHUNKS,
      COUNT(DISTINCT CHUNK_INDEX) AS DISTINCT_CHUNKS,MIN(CHUNK_INDEX) AS FIRST_CHUNK,MAX(CHUNK_INDEX) AS LAST_CHUNK,
      COUNT(DISTINCT SHA256) AS HASH_COUNT,SUM(LENGTH(PAYLOAD_BASE64)) AS ENCODED_BYTES
      FROM {qi(table)} GROUP BY ARTIFACT_NAME ORDER BY ARTIFACT_NAME""",max_rows=100)
    records=inventory.to_dict('records')
    artifact_inventory.extend({'table':table,**r} for r in records)
    emit('E10.inventory.'+table.rsplit('.',1)[-1], 'READ_OK',
         'Artifact names and sizes only; table timestamps do not establish run chronology.', tables=records,
         details={'source':table, 'catalog_created':str(catalog_record.get('CREATED')),
                  'catalog_last_altered':str(catalog_record.get('LAST_ALTERED'))})
    if not READ_SAFE_ARTIFACT_REPORTS:
        return
    for item in records:
        name=str(item['ARTIFACT_NAME'])
        if name not in SAFE_REPORT_NAMES:
            continue
        def load_one(name=name, item=item):
            if not (0 <= int(item['MIN_BYTES']) <= int(item['MAX_BYTES']) <= MAX_ARTIFACT_BYTES):
                emit('E10.report.'+table.rsplit('.',1)[-1]+'.'+name, 'SKIPPED', 'Allowed report exceeds the configured byte bound.',
                     details={'source':table,'artifact':name,'bytes':int(item['MAX_BYTES'])})
                return
            if int(item['ENCODED_BYTES']) > 4*((MAX_ARTIFACT_BYTES+2)//3):
                raise ValueError('Stored encoded payload exceeds byte bound.')
            chunk_count=int(item['MAX_CHUNKS'])
            if not 1 <= chunk_count <= 10000:
                raise ValueError('Artifact chunk count outside safe bound.')
            frame=run_query('SELECT '+','.join(qi(c) for c in sorted(ARTIFACT_COLUMNS))+' FROM '+qi(table)+
                            ' WHERE ARTIFACT_NAME='+lit(name)+' ORDER BY CHUNK_INDEX',max_rows=chunk_count)
            blob=decode_checked_artifact(frame,MAX_ARTIFACT_BYTES)
            safe_rows,omitted=extract_safe_report(blob,name)
            record={'source':table,'artifact':name,'sha256':hashlib.sha256(blob).hexdigest(),
                    'integrity_verified':True,'fields':safe_rows,'omitted_fields_or_records':omitted}
            recovered_safe_reports.append(record)
            emit('E10.report.'+table.rsplit('.',1)[-1]+'.'+name, 'READ_OK' if safe_rows else 'INCONCLUSIVE',
                 'Integrity-verified saved aggregate report; comparison validity has not been established.',
                 tables=safe_rows, details={k:v for k,v in record.items() if k!='fields'})
        attempt('E10.report.'+table.rsplit('.',1)[-1]+'.'+name,load_one)

def collect_saved_history():
    catalog=discovered_tables.copy()
    catalog.columns=[str(c).upper() for c in catalog.columns]
    if catalog.empty or not {'TABLE_NAME','TABLE_SCHEMA'} <= set(catalog.columns):
        emit('E10','UNAVAILABLE','No catalog inventory is available; run discovery first.')
        return
    # Scope is this project, not every accessible artifact table.
    catalog=catalog[catalog.TABLE_NAME.astype(str).str.startswith('TAK861_TX_READY_V63_DL_POC_')]
    eligible=[]
    for rec in catalog.to_dict('records'):
        table='DSVC_TAKEDA_TA_PRIVATE.'+str(rec['TABLE_SCHEMA'])+'.'+str(rec['TABLE_NAME'])
        # TABLE_SCHEMA/COLUMN_SCHEMA are metadata only. Query schema to identify real artifact tables.
        try:
            if ARTIFACT_COLUMNS <= columns(table):
                eligible.append((table,rec))
        except Exception:
            # Main connector helpers suppress sensitive exception text; report no raw exception here.
            emit('E10.schema.'+table.rsplit('.',1)[-1],'UNAVAILABLE','Artifact-table schema could not be read.',details={'source':table})
    # Prioritize documented reporting workflows before prepared/split blobs. This is not chronology.
    def report_priority(item):
        name=item[0]
        reporting=any(token in name for token in ('_MODEL','_EVAL','_SELECTION','_INVESTIGATION'))
        if name.endswith(('_MODEL_RUN_001','_EVAL_RUN_001')):
            rank=0
        elif '_SNAPSHOT_PAD_' in name and reporting:
            rank=1
        elif '_TEMPORAL_CLEAN_' in name and reporting:
            rank=2
        elif reporting:
            rank=3
        else:
            rank=4
        return rank,name
    eligible.sort(key=report_priority)
    prepared=[item for item in eligible if item[0].endswith(('_INPUTS','_PREPARED'))]
    historical=[item for item in eligible if item not in prepared]
    prepared_cap=globals().get('MAX_PREPARED_TABLES',4)
    if isinstance(prepared_cap,bool) or not isinstance(prepared_cap,int) or not 0<=prepared_cap<=50:
        raise ValueError('MAX_PREPARED_TABLES must be an integer from 0 to 50.')
    # Use a separate cap so input contracts do not displace historical model summaries.
    prepared.sort(key=lambda item:(0 if '_SNAPSHOT_PAD_' in item[0] else 1 if '_TEMPORAL_CLEAN_' in item[0] else 2,item[0]))
    chosen=historical[:MAX_ARTIFACT_TABLES]+prepared[:prepared_cap]
    emit('E10','READ_OK','Read-only recovery scope; this ordering is not experiment chronology.',
         details={'eligible_artifact_tables':len(eligible),'selected_tables':len(chosen),
                  'selected_history_tables':min(len(historical),MAX_ARTIFACT_TABLES),
                  'selected_prepared_tables':min(len(prepared),prepared_cap),
                  'omitted_history_tables':[t for t,_ in historical[MAX_ARTIFACT_TABLES:]],
                  'omitted_prepared_tables':[t for t,_ in prepared[prepared_cap:]],
                  'table_cap':MAX_ARTIFACT_TABLES,'prepared_table_cap':prepared_cap,
                  'payload_reading_enabled':READ_SAFE_ARTIFACT_REPORTS})
    for table,rec in chosen:
        attempt('E10.table.'+table.rsplit('.',1)[-1],lambda t=table,r=rec:collect_one_artifact_table(t,r))
attempt('E10',collect_saved_history)


## E11 — Candidate holdout metadata, with honest limits

**Why:** We need to determine whether another evaluation population might exist without using its outcomes to tune the experiment. This cell inventories known candidate snapshot/scored tables, counts patients and snapshot keys outside the frozen cohort, and identifies later rows with valid labels and a 90-day calendar window ending by the documented 2026-08-25 source vintage. That date is an explicit calendar assumption, not verified outcome-data completeness. The frozen snapshot table is included as a baseline check. Available V63 feature columns are counted when the 49-feature list was recovered. It creates no predictions or performance metrics.

**Output meaning:** A later date range or non-overlapping patient keys is only a candidate-population lead. It cannot prove that labels have matured, that claims are complete, or that the population was excluded from past fitting, preprocessing, feature selection and human inspection. `candidate_possible` only means some rows meet the stated new-patient/later-date/calendar/label rule. It is not permission to use those rows as a final test set, and false does not rule out every other valid holdout design. Historical-use and actual-maturity fields remain `UNKNOWN` and belong in the evidence ledger in E12.

**Genie help:** “Identify any already-approved population with complete next-90-day outcomes and documented exclusion from all previous fitting/selection/evaluation. Cite the population definition, outcome data-completeness date and access/use history. Do not infer ‘untouched’ from a date range or a different table name.”

In [ ]:

holdout_candidate_metadata = []
def collect_holdout_metadata():
    candidates=[]
    for key in ('snapshots','backtest_scored','patients_scored'):
        table=tables.get(key)
        if isinstance(table,str) and table not in candidates:
            candidates.append(table)
    frozen_table=tables.get('frozen_split')
    frozen_available=isinstance(frozen_table,str) and {'PATIENT_ID','END_DT'} <= columns(frozen_table)
    known_features=globals().get('evidence_v63_features')
    valid_feature_list=(isinstance(known_features,list) and len(known_features)==49
                        and all(isinstance(name,str) for name in known_features)
                        and len(set(known_features))==49)
    # This is a documented source-vintage calendar bound, NOT verified claims completeness.
    calendar_bound='2026-08-25'
    for table in candidates:
        def inspect(table=table):
            cols=columns(table)
            if not {'PATIENT_ID','END_DT'} <= cols:
                emit('E11.candidate.'+table.rsplit('.',1)[-1],'INCONCLUSIVE','Confirmed patient/date key columns are unavailable.',
                     details={'source':table,'columns':sorted(cols)})
                return
            feature_count=sum(name.upper() in cols for name in known_features) if valid_feature_list else None
            if not frozen_available:
                emit('E11.candidate.'+table.rsplit('.',1)[-1],'INCONCLUSIVE','Frozen cohort keys unavailable; candidate exclusion cannot be assessed.',
                     details={'source':table,'resp_column_available':'RESP' in cols,'available_v63_feature_columns':feature_count})
                return
            has_resp='RESP' in cols
            label_expression='s.RESP' if has_resp else 'CAST(NULL AS NUMBER)'
            query=f"""WITH frozen_keys AS (
              SELECT DISTINCT PATIENT_ID,END_DT FROM {qi(frozen_table)}
            ), frozen_patients AS (
              SELECT DISTINCT PATIENT_ID FROM {qi(frozen_table)} WHERE PATIENT_ID IS NOT NULL
            ), frozen_bounds AS (
              SELECT MAX(TRY_TO_DATE(TO_VARCHAR(END_DT))) AS MAX_FROZEN_END_DT FROM {qi(frozen_table)}
            ), candidate AS (
              SELECT s.PATIENT_ID,s.END_DT,TRY_TO_DATE(TO_VARCHAR(s.END_DT)) AS SNAPSHOT_DATE,
                     {label_expression} AS LABEL,
                     k.PATIENT_ID IS NULL AS SNAPSHOT_OUTSIDE_FROZEN,
                     p.PATIENT_ID IS NULL AS PATIENT_OUTSIDE_FROZEN,b.MAX_FROZEN_END_DT
              FROM {qi(table)} s
              LEFT JOIN frozen_keys k ON k.PATIENT_ID=s.PATIENT_ID AND k.END_DT=s.END_DT
              LEFT JOIN frozen_patients p ON p.PATIENT_ID=s.PATIENT_ID
              CROSS JOIN frozen_bounds b
            ), flags AS (
              SELECT *,NULLIF(TRIM(TO_VARCHAR(PATIENT_ID)), '') IS NOT NULL AND SNAPSHOT_DATE IS NOT NULL AS VALID_KEY,
                     SNAPSHOT_DATE>MAX_FROZEN_END_DT AS LATER_THAN_FROZEN_MAX,
                     DATEADD('day',90,SNAPSHOT_DATE)<=TO_DATE({lit(calendar_bound)}) AS CALENDAR_90D_ELIGIBLE
              FROM candidate
            )
            SELECT COUNT(*) AS N_ROWS,COUNT(DISTINCT PATIENT_ID) AS PATIENTS,
              COUNT(DISTINCT PATIENT_ID,END_DT) AS SNAPSHOT_KEYS,
              MIN(SNAPSHOT_DATE) AS MIN_END_DT,MAX(SNAPSHOT_DATE) AS MAX_END_DT,
              MAX(MAX_FROZEN_END_DT) AS MAX_FROZEN_END_DT,
              COALESCE(COUNT_IF(NOT VALID_KEY),0) AS INVALID_KEYS_OR_DATES,
              COUNT(DISTINCT CASE WHEN VALID_KEY AND PATIENT_OUTSIDE_FROZEN THEN PATIENT_ID END) AS PATIENTS_OUTSIDE_FROZEN,
              COUNT(DISTINCT CASE WHEN VALID_KEY AND SNAPSHOT_OUTSIDE_FROZEN THEN PATIENT_ID END,
                             CASE WHEN VALID_KEY AND SNAPSHOT_OUTSIDE_FROZEN THEN END_DT END) AS SNAPSHOTS_OUTSIDE_FROZEN,
              COALESCE(COUNT_IF(VALID_KEY AND LATER_THAN_FROZEN_MAX),0) AS LATER_DATED_ROWS,
              COALESCE(COUNT_IF(LABEL IS NULL OR LABEL NOT IN (0,1)),0) AS INVALID_OR_MISSING_LABEL_ROWS,
              COALESCE(COUNT_IF(VALID_KEY AND PATIENT_OUTSIDE_FROZEN AND LATER_THAN_FROZEN_MAX AND CALENDAR_90D_ELIGIBLE),0) AS NEW_PATIENT_LATER_CALENDAR_ELIGIBLE_ROWS,
              COUNT(DISTINCT CASE WHEN VALID_KEY AND PATIENT_OUTSIDE_FROZEN AND LATER_THAN_FROZEN_MAX
                                      AND CALENDAR_90D_ELIGIBLE AND LABEL IN (0,1) THEN PATIENT_ID END,
                             CASE WHEN VALID_KEY AND PATIENT_OUTSIDE_FROZEN AND LATER_THAN_FROZEN_MAX
                                      AND CALENDAR_90D_ELIGIBLE AND LABEL IN (0,1) THEN END_DT END) AS POTENTIAL_CANDIDATE_SNAPSHOTS
            FROM flags"""
            rows=run_query(query,max_rows=1).to_dict('records')
            if not has_resp:
                for row in rows:
                    row['INVALID_OR_MISSING_LABEL_ROWS']=None
                    row['POTENTIAL_CANDIDATE_SNAPSHOTS']=None
            eligible=rows[0].get('POTENTIAL_CANDIDATE_SNAPSHOTS') if rows else None
            possible=bool(eligible>0) if has_resp and eligible is not None and pd.notna(eligible) else None
            record={'source':table,'metadata':rows,'outcome_horizon_days':90,
                    'resp_column_available':has_resp,'available_v63_feature_columns':feature_count,
                    'calendar_bound_assumption':calendar_bound,
                    'calendar_bound_meaning':'Documented source vintage only; not certified outcome-data completeness.',
                    'candidate_possible':possible,
                    'candidate_rule':'New patient outside frozen cohort AND later than frozen maximum END_DT AND valid binary label AND END_DT+90<=calendar bound.',
                    'candidate_rule_is_not_holdout_certification':True,
                    'claims_complete_through':'UNKNOWN','mature_outcomes_verified':'UNKNOWN',
                    'excluded_from_model_and_preprocessing_fit':'UNKNOWN',
                    'excluded_from_prior_human_inspection':'UNKNOWN','untouched_holdout_status':'UNKNOWN'}
            holdout_candidate_metadata.append(record)
            emit('E11.candidate.'+table.rsplit('.',1)[-1],'READ_OK','Candidate counts use a calendar assumption only; historical fitting, prior use and actual outcome completeness remain unverified.',
                 tables=rows,details={k:v for k,v in record.items() if k!='metadata'})
        attempt('E11.candidate.'+table.rsplit('.',1)[-1],inspect)
    emit('E11','INCONCLUSIVE','No candidate has been certified untouched by this notebook.',
         details={'candidates_inspected':len(holdout_candidate_metadata),'new_scoring_or_metrics':False})
attempt('E11',collect_holdout_metadata)


## E12 — Human/Genie evidence that metadata cannot certify

**Why:** a current table, matching tensor or current procedure body cannot prove
which version historically ran or which patients influenced fitting/development.
This ledger accepts source-backed answers without turning them into automatic PASS.

Fill only what you know, with a notebook revision, job/run audit, recorded owner
answer or other precise reference. Leave unavailable facts UNKNOWN or UNAVAILABLE.
Do not include patient IDs, passwords or full source dumps. You can also return
the answers separately with the cell outputs; editing this cell is optional.

Human-only confirmations (unless recoverable audit records establish them):
- Practical session runtime/compute restrictions for final tuning.
- Which original1,028 builder/version actually ran, including counting, mappings,
  cutoff and exported time order; or confirmation that the original is unavailable.
- Historical V63 cutoff/claims-lag and negative-label follow-up rules, plus the
  population used to fit feature selection and VALUE_P/VAR_TYP.
- Original LightGBM training/tuning/refit membership relative to the frozen split.
- Whether historical TEST results influenced feature, architecture, threshold or
  hyperparameter choices. TEST has already been viewed; this asks about decision use.
- Whether any candidate confirmation cohort was ever used for training, feature
  selection, threshold tuning, back-testing or other development; “none known” is valid.

**Genie prompt:** “Use the exact object hashes/revisions and E08 evidence to find
execution-bound source or fitting manifests for these questions. Cite the evidence
for each answer. Distinguish the current definition from the historical executed
version. If logs/source cannot establish a fact, say UNKNOWN and identify the owner
who could confirm it. Do not infer unused holdout from date/key separation.”


In [ ]:
human_evidence = {
    "runtime_budget": {"status": "UNKNOWN", "answer": "", "reference": ""},
    "original_1028_build": {"status": "UNKNOWN", "answer": "", "reference": ""},
    "v63_cutoff_and_label_followup": {"status": "UNKNOWN", "answer": "", "reference": ""},
    "v63_selection_and_encoding_fit_scope": {"status": "UNKNOWN", "answer": "", "reference": ""},
    "original_lightgbm_fit_scope": {"status": "UNKNOWN", "answer": "", "reference": ""},
    "historical_test_decision_use": {"status": "UNKNOWN", "answer": "", "reference": ""},
    "candidate_holdout_prior_use": {"status": "UNKNOWN", "answer": "", "reference": ""},
}

def human_ledger():
    rows = []
    for item, value in human_evidence.items():
        if value.get("status") not in {"UNKNOWN", "UNAVAILABLE", "PROVIDED"}:
            raise ValueError("Human entries may be UNKNOWN, UNAVAILABLE or PROVIDED, never automatic PASS.")
        if value["status"] == "PROVIDED" and (not value.get("answer", "").strip() or not value.get("reference", "").strip()):
            raise ValueError("A PROVIDED answer needs a precise source or owner confirmation reference.")
        rows.append({"item": item, **value})
    emit("E12", "USER_REPORTED" if any(r["status"] == "PROVIDED" for r in rows) else "INCONCLUSIVE",
         "Declared evidence awaits review; this cell cannot certify historical facts.", tables=rows)

attempt("E12", human_ledger)


## E13 — Return checklist and evidence limits

**Why:** assemble the completed, failed, skipped and unresolved sections so Codex
can consume your outputs in order without asking for information already collected.

**Return:** this checklist plus all earlier report pages. A section may have several
subreports. Missing sections remain NOT_RUN; running this last cell is not proof
the previous cells completed. Re-run E00 onward after changing sources/configuration
so one run ID describes one coherent pass. No production conclusions are made here.

Next, Codex will review the outputs and human confirmations, resolve input-specific
limitations, and update the existing four model notebooks plus the fifth comparison
notebook. This auxiliary evidence collector is not a sixth final modeling workflow.


In [ ]:
def evidence_checklist():
    expected = ["E%02d" % i for i in range(13)]
    rows = []
    for prefix in expected:
        found = [r for r in reports if re.match(re.escape(prefix) + r"(?:[A-Z]?[_.]|$)", r["section"])]
        if not found:
            rows.append({"section": prefix, "status": "NOT_RUN", "summary": "No report recorded in this run."})
        else:
            rows.extend({"section": r["section"], "status": r["status"], "summary": r["summary"]} for r in found)
    emit("E13", "INCONCLUSIVE", "Evidence collection summary; this is not a model validation or a leakage certificate.", tables=rows,
         details={"query_count": len(query_audit), "query_failures": sum(x["status"] != "READ_OK" for x in query_audit),
                  "query_seconds": sum(x["elapsed_seconds"] for x in query_audit),
                  "runtime_budget_needed": True,
                  "untouched_test_claim": False,
                  "next_step": "Return reports and any human/source confirmations. Missing history has an explicit unavailable disposition; no metric is invented."})

evidence_checklist()
